# CryoBoltz ❄️⚡ &nbsp;—&nbsp; run it in Google Colab

**CryoBoltz** fits atomic structures into cryo-EM density maps of *dynamic* proteins. It steers the
[Boltz-1](https://github.com/jwohlwend/boltz) diffusion trajectory at inference time with a two-stage
guidance scheme (a coarse *global* stage that pulls the structure into the map, then a *local* stage
that refines the fit against the map's forward model), so it can recover conformations that differ
from the AlphaFold/Boltz consensus prediction.

[📄 Paper](https://arxiv.org/abs/2506.04490) · [🌐 Project page](https://cryoboltz.cs.princeton.edu/) · [💻 GitHub](https://github.com/ml-struct-bio/cryoboltz)

---

### What you need

| # | Input | Format | Notes |
|---|-------|--------|-------|
| 1 | **Sequence** | FASTA or YAML | Boltz conventions: `>A\|protein` per chain. Guidance is applied to **protein** atoms (H, C, N, O, S) only. |
| 2 | **Density map** | MRC / CCP4 (`.mrc`, `.map`) | Cubic voxels, standard axis order. You also want an estimated **resolution** and a **contour threshold** that excludes background. |
| 3 | **Roughly aligned structure** | mmCIF | Used only to place the prediction in the map's frame before guidance starts — *not* used as a template. Chain labels must match the sequence file. |

Don't have #3? **Step 6** can build one for you (unguided Boltz prediction + rigid-body fit into the map),
so you never have to leave the notebook. Already have a model aligned to your maps — in ChimeraX, say, or to
**cryoDRGN** volumes? Step 6 will prepare it without moving a single atom, and **Step 3b** handles cryoDRGN's
conventions (including its `--Apix` default of 1.0).

---

### How this notebook is laid out

| Step | What it does | Typical time |
|------|--------------|--------------|
| 0 | Check the runtime (GPU / RAM / disk) | seconds |
| 1 | Install CryoBoltz + load helpers | 4–8 min, once per session |
| 2 | Download model weights (≈2.2 GB), optionally cached on Drive | 1–4 min |
| 3 | Choose inputs — built-in smoke test, the paper's Pma1 example, your own uploads, or PDB/EMDB IDs | 0–10 min |
| 3b | *Optional:* **cryoDRGN** volumes — collect an ensemble, check its pixel size | seconds |
| 4 | **Validate** sequence ↔ structure ↔ map, pick a threshold, size the run | seconds |
| 5 | Crop / bin the map (big speed + memory win) | seconds |
| 6 | *Optional:* prepare, build and rigid-body fit the initial structure | 0–20 min |
| 7 | **Run CryoBoltz** | minutes → hours |
| 7b | *Optional:* **batch** — one run per volume, MSA computed once | N × Step 7 |
| 8 | Confidence scores + guidance diagnostics | seconds |
| 9 | 3D visualisation (model in map) | seconds |
| 10 | Download / save results | seconds |

> ⚙️ **Before you start:** `Runtime ▸ Change runtime type ▸ GPU`. A free **T4 (15 GB)** is enough for the
> built-in smoke test and small systems (≲300 residues). Larger systems — including the Pma1 example from
> the paper (918 residues) — need an **L4 / A100**, because guidance runs the network with gradients
> enabled in float32.
>
> 💡 Run the cells top to bottom. Every cell is a form: double-click the title (or click ⌄) to reveal the code.

## Step 0 · Check the runtime

This tells you what hardware Colab gave you and what it can realistically fit.

In [ ]:
#@title Step 0 · Check GPU / RAM / disk { display-mode: "form" }
import os, shutil, subprocess, sys

def _sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=90).stdout.strip()
    except Exception:
        return ""

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
ROOT = "/content" if os.path.isdir("/content") else os.getcwd()

gpu_line = _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
VRAM_GB = 0.0
print("Python  :", sys.version.split()[0])
if gpu_line:
    name, mem, drv = (x.strip() for x in gpu_line.split(",")[:3])
    VRAM_GB = float(mem.split()[0]) / 1024.0
    print(f"GPU     : {name}  —  {VRAM_GB:.1f} GiB VRAM, driver {drv}")
else:
    print("GPU     : none detected  →  Runtime ▸ Change runtime type ▸ GPU, then re-run this cell.")
    print("          (CryoBoltz will run on CPU but is far too slow to be useful.)")

try:
    RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
except (ValueError, OSError):
    RAM_GB = float("nan")
DISK_GB = shutil.disk_usage(ROOT).free / 1024**3
print(f"RAM     : {RAM_GB:.1f} GiB")
print(f"Disk    : {DISK_GB:.1f} GiB free at {ROOT}")
print(f"Colab   : {'yes' if IN_COLAB else 'no (Drive / upload / download cells will be skipped)'}")

print("\nRough capacity guide (guidance needs gradients + float32, so it is memory hungry):")
if VRAM_GB == 0:
    print("  • no GPU            → only the smoke test will finish, and slowly")
elif VRAM_GB < 17:
    print(f"  • {VRAM_GB:.0f} GiB (e.g. T4)   → smoke test ✔ · up to roughly 250–350 residues · crop the map hard")
elif VRAM_GB < 30:
    print(f"  • {VRAM_GB:.0f} GiB (e.g. L4)   → up to roughly 500–700 residues")
else:
    print(f"  • {VRAM_GB:.0f} GiB (A100/H100) → ~1000 residues, i.e. the Pma1 example from the paper")
print("  These are ballpark numbers: memory also scales with --diffusion_samples, map size and MSA depth.")
if DISK_GB < 8:
    print("\n⚠️  Less than 8 GiB of disk free. Weights need ~2.5 GiB and a raw EMDB map can be ~1 GiB.")

## Step 1 · Install CryoBoltz

Clones the repository and installs it with `pip install -e .`, exactly as in the README. Both cells below
are safe to re-run — they skip work that is already done. Point `REPO_URL` / `BRANCH` at your own fork if you
want to run modified code.

> The install pins `numpy==1.26.3`, which downgrades Colab's NumPy. That is expected. If a later cell ever
> raises a NumPy ABI error (`module compiled against NumPy 2.x…`), do `Runtime ▸ Restart session` and re-run
> from Step 1 — the packages stay installed, so it only takes seconds.

In [ ]:
#@title Step 1 · Clone + install (4–8 min, once per session) { display-mode: "form" }
REPO_URL = "https://github.com/ml-struct-bio/cryoboltz.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
WORK_DIR = "/content/cryoboltz_work" #@param {type:"string"}
FORCE_REINSTALL = False #@param {type:"boolean"}

import importlib, importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

WORK_DIR = Path(WORK_DIR)
REPO_DIR = WORK_DIR / "cryoboltz"
LOG_DIR = WORK_DIR / "logs"
for d in (WORK_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

INTERESTING = ("Collecting ", "Downloading ", "Building ", "Installing ", "Successfully ",
               "ERROR", "error:", "Killed", "Cloning ")

def _run(cmd, cwd=None, log=None, quiet_filter=True):
    """Run a command, echo the interesting lines, keep the full output in `log`."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd) if cwd else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors="replace", bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not quiet_filter or line.startswith(INTERESTING):
            print("  " + line.rstrip()[:160])
    rc = proc.wait()
    if log:
        Path(log).write_text("".join(lines))
    if rc != 0:
        print("\n".join(lines[-40:]))
        raise RuntimeError(f"command failed (exit {rc}); full log: {log}")
    return rc

t0 = time.time()

if (REPO_DIR / "pyproject.toml").exists():
    print(f"✔ repository already at {REPO_DIR}")
else:
    _run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
         log=LOG_DIR / "clone.log")

already = importlib.util.find_spec("boltz") is not None
if already and not FORCE_REINSTALL:
    print("✔ boltz already importable — skipping pip (tick FORCE_REINSTALL to redo it)")
else:
    print("\nInstalling — this pulls ~1 GB of wheels, please be patient…")
    _run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO_DIR,
         log=LOG_DIR / "pip_install.log")
    _run([sys.executable, "-m", "pip", "install", "py3Dmol"], log=LOG_DIR / "pip_extras.log")

# An editable install drops a .pth file that this already-running kernel never read, so make the
# package importable right now instead of asking for a restart.
import site
importlib.invalidate_caches()
for path in site.getsitepackages() + [site.getusersitepackages()]:
    if os.path.isdir(path):
        site.addsitedir(path)
src_dir = str((REPO_DIR / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# `boltz predict` resolves relative MSA paths (e.g. in examples/prot.fasta) against the cwd
os.chdir(REPO_DIR)

# prefer the console script, fall back to `python -m` if it is not on PATH
BOLTZ_CMD = ["boltz"] if shutil.which("boltz") else [sys.executable, "-m", "boltz.main"]
show = subprocess.run([sys.executable, "-m", "pip", "show", "boltz"], capture_output=True, text=True).stdout
version = next((l.split(":", 1)[1].strip() for l in show.splitlines() if l.startswith("Version")), "?")
cli_ok = subprocess.run([*BOLTZ_CMD, "predict", "--help"], capture_output=True, text=True)
importable = importlib.util.find_spec("boltz") is not None

print(f"\n✔ boltz {version} · `{' '.join(BOLTZ_CMD)} predict --help` exit code {cli_ok.returncode}")
print(f"✔ importable in this kernel: {importable}")
print(f"✔ working directory   : {os.getcwd()}")
print(f"✔ scratch / outputs in: {WORK_DIR}")
print(f"⏱  step 1 took {time.time() - t0:.0f} s")
if cli_ok.returncode != 0:
    print("\n⚠️  The CLI did not run. Output:\n", cli_ok.stdout[-2000:], cli_ok.stderr[-2000:])
    print("Try `Runtime ▸ Restart session` and re-run this cell — the install itself is already done.")

### Step 1b · Load this notebook's helper functions

Everything the notebook adds on top of the CryoBoltz CLI lives here: map bookkeeping (headers, origins,
thresholds, binning), structure parsing, a synthetic-map generator for the smoke test, a rigid-body fitter
that stands in for ChimeraX `fitmap`, and the plotting utilities. **Re-run this cell after any session restart.**

In [ ]:
#@title Step 1b · Helper functions (run once per session) { display-mode: "form" }
import json, os, re, selectors, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

import gemmi
import mrcfile
import numpy as np

# ─────────────────────────────────────────────────────────── misc plumbing

def require(*names):
    """Fail with a helpful message when an earlier step has not been run."""
    missing = [n for n in names if n not in globals() or globals()[n] is None]
    if missing:
        raise RuntimeError(f"{', '.join(missing)} not set — run the earlier steps first.")

def human(n_bytes):
    for unit in ("B", "KiB", "MiB", "GiB"):
        if abs(n_bytes) < 1024 or unit == "GiB":
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024.0

def run_streaming(cmd, cwd=None, log_path=None, env=None):
    """Run a command, mirroring its output live (progress bars included) into a log file."""
    cmd = [str(c) for c in cmd]
    child_env = dict(os.environ, PYTHONUNBUFFERED="1")
    if env:
        child_env.update(env)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, bufsize=0, env=child_env)
    sink = open(log_path, "wb") if log_path else None
    sel = selectors.DefaultSelector()
    sel.register(proc.stdout, selectors.EVENT_READ)
    try:
        while True:
            for key, _ in sel.select(timeout=0.2):
                data = os.read(key.fd, 65536)
                if not data:
                    continue
                sys.stdout.write(data.decode("utf-8", "replace"))
                sys.stdout.flush()
                if sink:
                    sink.write(data)
                    sink.flush()
            if proc.poll() is not None:
                rest = proc.stdout.read() or b""
                if rest:
                    sys.stdout.write(rest.decode("utf-8", "replace"))
                    if sink:
                        sink.write(rest)
                break
    finally:
        sel.close()
        proc.stdout.close()
        if sink:
            sink.close()
    return proc.wait()

def download(url, dst, expect_gz=False):
    """Download `url` to `dst` (resumable via wget when available). Returns the path."""
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        print(f"  ✔ already have {dst.name} ({human(dst.stat().st_size)})")
        return dst
    tmp = dst.with_suffix(dst.suffix + ".part")
    if shutil.which("wget"):
        rc = subprocess.run(["wget", "-q", "--show-progress", "--progress=bar:force:noscroll",
                             "-c", "-O", str(tmp), url]).returncode
    else:
        rc = subprocess.run(["curl", "-fL", "-C", "-", "-o", str(tmp), url]).returncode
    if rc != 0 or not tmp.exists() or tmp.stat().st_size == 0:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f"download failed: {url}")
    tmp.rename(dst)
    print(f"  ✔ {dst.name} ({human(dst.stat().st_size)})")
    return dst

def download_first(urls, dst):
    """Try several mirrors in order."""
    errors = []
    for url in urls:
        try:
            print(f"  → {url}")
            return download(url, dst)
        except Exception as exc:  # noqa: BLE001
            errors.append(f"{url}: {exc}")
    raise RuntimeError("all mirrors failed:\n  " + "\n  ".join(errors))

def gunzip(src, dst=None):
    import gzip
    src = Path(src)
    dst = Path(dst) if dst else src.with_suffix("")
    if dst.exists() and dst.stat().st_size > 0:
        print(f"  ✔ already unpacked {dst.name}")
        return dst
    with gzip.open(src, "rb") as fi, open(dst, "wb") as fo:
        shutil.copyfileobj(fi, fo, length=8 << 20)
    print(f"  ✔ unpacked {dst.name} ({human(dst.stat().st_size)})")
    return dst

# ─────────────────────────────────────────────────────────── sequence files

ENTITY_KEYS = ("protein", "dna", "rna", "ligand")

def parse_sequence_file(path):
    """[(chain_id, entity_type, sequence_or_code), ...] from a Boltz FASTA or YAML."""
    path = Path(path)
    if path.suffix.lower() in (".yaml", ".yml"):
        import yaml
        spec = yaml.safe_load(path.read_text())
        out = []
        for item in spec.get("sequences", []):
            for key, body in item.items():
                if key not in ENTITY_KEYS:
                    continue
                ids = body.get("id")
                ids = ids if isinstance(ids, list) else [ids]
                seq = body.get("sequence") or body.get("ccd") or body.get("smiles") or ""
                for cid in ids:
                    out.append((str(cid), key, seq))
        return out
    out, cid, etype, buf = [], None, None, []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cid is not None:
                out.append((cid, etype, "".join(buf)))
            fields = line[1:].split("|")
            cid = fields[0].strip()
            etype = fields[1].strip().lower() if len(fields) > 1 else "protein"
            etype = {"ccd": "ligand", "smiles": "ligand"}.get(etype, etype)
            buf = []
        else:
            buf.append(line)
    if cid is not None:
        out.append((cid, etype, "".join(buf)))
    return out

def write_protein_only_sequence(src, dst):
    """Copy a Boltz FASTA/YAML keeping only its protein entities. Returns (kept, dropped) chain lists.

    Guidance is protein-only, so a sequence file listing ligands, metals or nucleic acids cannot be used for
    a guided run. Constraints that reference a dropped chain go with it.
    """
    src, dst = Path(src), Path(dst)
    kept, dropped = [], []
    if src.suffix.lower() in (".yaml", ".yml"):
        import yaml
        spec = yaml.safe_load(src.read_text())
        entries = []
        for item in spec.get("sequences", []):
            key = next(iter(item))
            ids = item[key].get("id")
            ids = [str(i) for i in (ids if isinstance(ids, list) else [ids])]
            (kept if key == "protein" else dropped).extend(ids)
            if key == "protein":
                entries.append(item)
        spec["sequences"] = entries
        if spec.get("constraints"):
            gone = set(dropped)
            keep_constraints = []
            for c in spec["constraints"]:
                body = next(iter(c.values()))
                refs = {str(body.get("binder"))} | {str(x[0]) for x in body.get("contacts", []) if x}
                refs |= {str(t) for t in body.get("token1", []) + body.get("token2", []) if not isinstance(t, int)}
                if refs & gone:
                    continue
                keep_constraints.append(c)
            if len(keep_constraints) != len(spec["constraints"]):
                print(f"    dropped {len(spec['constraints']) - len(keep_constraints)} constraint(s) that "
                      "referenced a removed chain")
            spec["constraints"] = keep_constraints
            if not keep_constraints:
                spec.pop("constraints")
        dst.write_text(yaml.safe_dump(spec, sort_keys=False))
    else:
        out, keep = [], False
        for line in src.read_text().splitlines():
            if line.startswith(">"):
                fields = line[1:].split("|")
                etype = fields[1].strip().lower() if len(fields) > 1 else "protein"
                keep = etype == "protein"
                (kept if keep else dropped).append(fields[0].strip())
            if keep:
                out.append(line)
        dst.write_text("\n".join(out) + "\n")
    return kept, dropped

def has_precomputed_msa(path):
    """True when every protein record already points at an MSA (so --use_msa_server is unnecessary)."""
    path = Path(path)
    if path.suffix.lower() in (".yaml", ".yml"):
        import yaml
        spec = yaml.safe_load(path.read_text())
        prots = [b for it in spec.get("sequences", []) for k, b in it.items() if k == "protein"]
        return bool(prots) and all(b.get("msa") for b in prots)
    prots = [l for l in path.read_text().splitlines() if l.startswith(">")]
    prots = [l for l in prots if len(l.split("|")) > 1 and l.split("|")[1].strip().lower() == "protein"]
    return bool(prots) and all(len(l.split("|")) > 2 and l.split("|")[2].strip() for l in prots)

def rcsb_fasta_to_boltz(text, out_path, keep_n_chains=0):
    """Rewrite an RCSB FASTA into Boltz's `>CHAIN|protein` convention. Returns [(chain, length)]."""
    entries, header, buf = [], None, []
    for line in text.splitlines():
        if line.startswith(">"):
            if header is not None:
                entries.append((header, "".join(buf)))
            header, buf = line[1:], []
        elif line.strip():
            buf.append(line.strip())
    if header is not None:
        entries.append((header, "".join(buf)))

    records = []
    for head, seq in entries:
        chains = ["A"]
        m = re.search(r"Chains?\s+([^|]+)", head)
        if m:
            chains = [c.strip().split("[")[0].strip() for c in m.group(1).split(",") if c.strip()]
        for c in chains:
            records.append((c, seq))
    if keep_n_chains:
        records = records[:keep_n_chains]
    seen, unique = set(), []
    for c, seq in records:
        while c in seen:
            c += "'"
        seen.add(c)
        unique.append((c, seq))
    Path(out_path).write_text("".join(f">{c}|protein\n{seq}\n" for c, seq in unique))
    return [(c, len(s)) for c, s in unique]

# ─────────────────────────────────────────────────────────── density maps

class MapInfo:
    """An MRC map read the way CryoBoltz reads it: cubic voxels, header `origin` only."""

    def __init__(self, path):
        self.path = str(path)
        with mrcfile.open(self.path, permissive=True, mode="r") as mrc:
            self.data = np.asarray(mrc.data, dtype=np.float32)
            self.voxel = (float(mrc.voxel_size.x), float(mrc.voxel_size.y), float(mrc.voxel_size.z))
            self.origin = np.array([float(mrc.header.origin.x), float(mrc.header.origin.y),
                                    float(mrc.header.origin.z)], dtype=np.float64)
            self.nstart = np.array([int(mrc.header.nxstart), int(mrc.header.nystart),
                                    int(mrc.header.nzstart)], dtype=np.int64)
            self.axes = (int(mrc.header.mapc), int(mrc.header.mapr), int(mrc.header.maps))
        self.nz, self.ny, self.nx = self.data.shape

    spacing = property(lambda self: self.voxel[0])          # what CryoBoltz uses for every axis
    n_voxels = property(lambda self: int(self.nx) * int(self.ny) * int(self.nz))
    voxel_volume = property(lambda self: self.voxel[0] * self.voxel[1] * self.voxel[2])

    @property
    def box(self):
        """(low, high) physical corners in Angstrom."""
        low = self.origin
        return low, low + np.array([self.nx * self.voxel[0], self.ny * self.voxel[1], self.nz * self.voxel[2]])

    def grid_axes(self):
        return (self.origin[0] + np.arange(self.nx) * self.voxel[0],
                self.origin[1] + np.arange(self.ny) * self.voxel[1],
                self.origin[2] + np.arange(self.nz) * self.voxel[2])

    def sample(self, xyz):
        """Trilinear map values at physical coordinates (N, 3)."""
        from scipy.ndimage import map_coordinates
        idx = (np.asarray(xyz, dtype=np.float64) - self.origin) / np.array(self.voxel)
        return map_coordinates(self.data, np.stack([idx[:, 2], idx[:, 1], idx[:, 0]]),
                               order=1, mode="constant", cval=0.0)

    def summary(self):
        lo, hi = self.box
        return (f"{self.nx}×{self.ny}×{self.nz} = {self.n_voxels:,} voxels · spacing "
                f"{self.voxel[0]:.3f}/{self.voxel[1]:.3f}/{self.voxel[2]:.3f} Å\n"
                f"    origin ({lo[0]:.1f}, {lo[1]:.1f}, {lo[2]:.1f}) Å → corner "
                f"({hi[0]:.1f}, {hi[1]:.1f}, {hi[2]:.1f}) Å\n"
                f"    values min {self.data.min():.4g} · max {self.data.max():.4g} · "
                f"mean {self.data.mean():.4g} · std {self.data.std():.4g}")

    def issues(self):
        """Things that would silently break or bias guidance."""
        msgs = []
        vx, vy, vz = self.voxel
        if vx <= 0:
            msgs.append("Voxel size is 0 in the header — set it before running (mrcfile: `mrc.voxel_size = s`).")
        elif max(abs(vx - vy), abs(vx - vz)) > 1e-3:
            msgs.append(f"Anisotropic voxels ({vx:.3f}, {vy:.3f}, {vz:.3f} Å). CryoBoltz uses the x spacing "
                        "for all three axes — resample to cubic voxels first.")
        if self.axes != (1, 2, 3):
            msgs.append(f"Non-standard axis order mapc/mapr/maps = {self.axes}; CryoBoltz assumes (1, 2, 3). "
                        "Permute the axes (e.g. `relion_image_handler` or ChimeraX `vop`) first.")
        if np.allclose(self.origin, 0) and np.any(self.nstart != 0):
            shift = self.nstart * np.array(self.voxel)
            msgs.append(f"Header origin is (0, 0, 0) but nstart is {tuple(int(v) for v in self.nstart)}, so the "
                        f"true origin is ({shift[0]:.2f}, {shift[1]:.2f}, {shift[2]:.2f}) Å. CryoBoltz reads only "
                        "`origin` — turn on FIX_ORIGIN in Step 5 to fold nstart in.")
        if not np.isfinite(self.data).all():
            msgs.append("Map contains NaN/Inf values — clean it before running.")
        return msgs

def write_map(data, path, spacing, origin):
    spacing = (spacing, spacing, spacing) if np.isscalar(spacing) else tuple(spacing)
    with mrcfile.new(str(path), overwrite=True) as mrc:
        mrc.set_data(np.ascontiguousarray(data, dtype=np.float32))
        mrc.voxel_size = spacing
        mrc.header.origin = tuple(float(v) for v in origin)
        mrc.header.nxstart, mrc.header.nystart, mrc.header.nzstart = 0, 0, 0
    return Path(path)

def fix_origin(src, dst):
    """Fold nstart into the header origin (and zero nstart)."""
    info = MapInfo(src)
    new_origin = info.origin + info.nstart * np.array(info.voxel)
    write_map(info.data, dst, info.voxel, new_origin)
    return new_origin

def flip_map(src, dst, axis="z"):
    """Mirror a map along one axis, keeping voxel size and origin so the physical box is unchanged.

    A z-flip is exactly what `cryodrgn eval_vol --flip` does (`vol.flip([0])` on a [z, y, x] array), i.e. the
    fix for a reconstruction of the wrong handedness. Mirroring invalidates any existing alignment: the
    density moves to the other side of the box, so re-fit the model afterwards.
    """
    info = MapInfo(src)
    slicer = {"z": (slice(None, None, -1), slice(None), slice(None)),
              "y": (slice(None), slice(None, None, -1), slice(None)),
              "x": (slice(None), slice(None), slice(None, None, -1))}
    if axis not in slicer:
        raise ValueError("axis must be 'x', 'y' or 'z'")
    return write_map(info.data[slicer[axis]], dst, info.voxel, info.origin)

def bin_map(src, dst, factor):
    """Block-average the map by an integer factor, keeping physical positions correct."""
    factor = int(factor)
    info = MapInfo(src)
    nz, ny, nx = (n // factor * factor for n in (info.nz, info.ny, info.nx))
    trimmed = info.data[:nz, :ny, :nx]
    binned = trimmed.reshape(nz // factor, factor, ny // factor, factor, nx // factor, factor).mean(axis=(1, 3, 5))
    # the centre of the first block sits half a block in from the old origin
    new_origin = info.origin + 0.5 * (factor - 1) * np.array(info.voxel)
    new_spacing = tuple(v * factor for v in info.voxel)
    return write_map(binned, dst, new_spacing, new_origin)

def set_voxel_size(src, dst, apix, keep_origin=True):
    """Rewrite a map's header pixel size (cryoDRGN's `--Apix` defaults to 1.0 and is often left there)."""
    info = MapInfo(src)
    origin = info.origin if keep_origin else np.zeros(3)
    write_map(info.data, dst, apix, origin)
    return Path(dst)

def crop_map(src, dst, repo_dir, thresh=None, pad=10, dim=None, verbose=True):
    """Crop a map with the repository's own scripts/preproc_map.py."""
    import subprocess, sys
    cmd = [sys.executable, str(Path(repo_dir) / "scripts/preproc_map.py"), str(src)]
    if dim:
        cmd += ["--dim", str(int(dim))]
    else:
        cmd += ["--thresh", str(float(thresh)), "--pad", str(int(pad))]
    cmd += ["-o", str(dst)]
    if verbose:
        print("   $", " ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout, out.stderr)
        raise RuntimeError("preproc_map.py failed (most often: no voxel is above --thresh)")
    return Path(dst)

def radius_of_gyration_map(info, thresh):
    """Intensity-weighted Rg of the density above `thresh`, in Angstrom."""
    w = np.clip(info.data - thresh, 0, None)
    zz, yy, xx = np.nonzero(w > 0)
    if not len(xx):
        return float("nan")
    weights = w[zz, yy, xx].astype(np.float64)
    pts = np.stack([xx * info.voxel[0], yy * info.voxel[1], zz * info.voxel[2]], axis=1)
    centre = (pts * weights[:, None]).sum(0) / weights.sum()
    return float(np.sqrt((weights * ((pts - centre) ** 2).sum(1)).sum() / weights.sum()))

def radius_of_gyration_model(path):
    """Z-weighted Rg of a structure, in Angstrom."""
    xyz, znum = all_atoms(path)
    centre = (xyz * znum[:, None]).sum(0) / znum.sum()
    return float(np.sqrt((znum * ((xyz - centre) ** 2).sum(1)).sum() / znum.sum()))

def estimate_pixel_size(info, model_path, thresh):
    """Compare map and model radii of gyration to sanity-check the header pixel size.

    The model is in Angstrom by construction, so if the header spacing were wrong by a factor s the
    density's Rg would be off by the same factor. Only meaningful when the map holds (roughly) the same
    matter as the model -- extra density, a partial model or a bad threshold all bias it.
    """
    rg_map = radius_of_gyration_map(info, thresh)
    rg_model = radius_of_gyration_model(model_path)
    scale = rg_model / rg_map if rg_map and np.isfinite(rg_map) else float("nan")
    return {"rg_map": rg_map, "rg_model": rg_model, "scale": scale,
            "implied_spacing": info.spacing * scale}

def expected_volume(n_residues, n_nucleotides=0):
    """Rough molecular volume: ≈133 Å³ per amino acid, ≈500 Å³ per nucleotide."""
    return 133.0 * n_residues + 500.0 * n_nucleotides

def pick_threshold(info, target_volume, verbose=True):
    """The contour level this notebook uses: volume-matched, rounded to 4 significant figures, never the floor."""
    value = float(f"{suggest_threshold(info, target_volume, verbose=verbose):.4g}")
    floor = float(info.data.min())
    floor += 1e-6 * (float(info.data.max()) - floor)      # "indistinguishable from background"
    if not np.isfinite(value) or value <= floor:
        value = float(f"{float(np.percentile(info.data, 97.5)):.4g}")
        if verbose:
            print(f"  ⚠️  the volume-matched level collapsed to the map's floor, which usually means the geometry is "
                  f"off (pixel size? box?). Using the 97.5th percentile instead: {value:g}")
    return value

def threshold_table(info, target_volume=None, percentiles=(50, 80, 90, 95, 97.5, 99, 99.5, 99.9)):
    data = info.data
    rows, seen = [], set()
    for p in percentiles:
        t = float(np.percentile(data, p))
        if t in seen or not (data.min() < t < data.max()):
            continue
        seen.add(t)
        n = int((data > t).sum())
        rows.append({"percentile": p, "threshold": t, "voxels_above": n,
                     "volume_A3": n * info.voxel_volume,
                     "vol / expected": (n * info.voxel_volume / target_volume) if target_volume else float("nan")})
    return rows

def suggest_threshold(info, target_volume, lo_pct=40.0, hi_pct=99.999, verbose=True):
    """Bisect for the contour level whose enclosed volume ≈ the expected molecular volume.

    When no level can enclose that much volume the map simply does not hold the molecule you think it does
    -- a wrong pixel size, a box that is too small, or only part of the complex -- so say so and fall back
    to a percentile instead of returning a meaningless floor value.
    """
    lo, hi = float(np.percentile(info.data, lo_pct)), float(np.percentile(info.data, hi_pct))
    if float((info.data > lo).sum()) * info.voxel_volume < target_volume:
        fallback = float(np.percentile(info.data, 97.5))
        if verbose:
            enclosed = float((info.data > lo).sum()) * info.voxel_volume
            print(f"  ⚠️  even at its {lo_pct:g}th percentile this map encloses only {enclosed:,.0f} Å³, well under "
                  f"the {target_volume:,.0f} Å³ your sequence needs. Check the header pixel size (a cryoDRGN "
                  f"default of 1.0?) and whether the map covers the whole complex. Falling back to a percentile "
                  f"threshold of {fallback:.5g}.")
        return fallback
    for _ in range(45):
        mid = 0.5 * (lo + hi)
        if float((info.data > mid).sum()) * info.voxel_volume > target_volume:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

# ─────────────────────────────────────────────────────────── structures

_ONE = {}

def _one_letter(resname):
    if resname not in _ONE:
        info = gemmi.find_tabulated_residue(resname)
        _ONE[resname] = info.one_letter_code.upper() if info else "X"
    return _ONE[resname]

def read_structure(path):
    st = gemmi.read_structure(str(path))
    st.setup_entities()
    st.merge_chain_parts()
    st.remove_waters()
    st.remove_hydrogens()
    st.remove_alternative_conformations()
    st.remove_empty_chains()
    return st

def cif_chains(path):
    """{chain: {'seq', 'ca' (N,3), 'label_seq', 'auth_seq', 'n_res'}} for model 1 of a structure file."""
    out = {}
    for chain in read_structure(path)[0]:
        seq, ca, lseq, auth = [], [], [], []
        for res in chain:
            code = _one_letter(res.name)
            if code in ("", " ", "X"):
                continue
            seq.append(code)
            lseq.append(res.label_seq)          # None when the file never set label_seq_id
            auth.append(res.seqid.num)
            atom = res.find_atom("CA", "*")
            if atom is not None:
                ca.append([atom.pos.x, atom.pos.y, atom.pos.z])
        if seq:
            out[chain.name] = {"seq": "".join(seq), "ca": np.array(ca, dtype=np.float64).reshape(-1, 3),
                               "label_seq": lseq, "auth_seq": auth, "n_res": len(seq)}
    return out

def label_seq_span(chain_info):
    """Printable label_seq range, tolerating files that never set label_seq_id."""
    vals = [v for v in chain_info["label_seq"] if v]
    if not vals:
        return "unset"
    return f"{min(vals)}–{max(vals)}"

def raw_structure_report(path):
    """Read a structure exactly as CryoBoltz's load_cif does — no entity setup — and report the traps.

    load_cif indexes residues by `label_seq`, which gemmi leaves unset for PDB files, and it offsets
    each chain by the *total* residue count of the previous chain, so a ligand sitting inside a protein
    chain shifts every later chain.
    """
    st = gemmi.read_structure(str(path))       # deliberately no setup_entities(), like load_cif
    st.merge_chain_parts()
    st.remove_waters()
    st.remove_hydrogens()
    report = {"chains": [], "label_seq_unset": 0, "aa_total": 0, "het_before_protein": [], "gaps": []}
    chains = []
    for chain in st[0]:
        aa, het, unset, auth = 0, [], 0, []
        for res in chain:
            info = gemmi.find_tabulated_residue(res.name)
            if info and info.is_amino_acid():
                aa += 1
                auth.append(res.seqid.num)
                if not res.label_seq:
                    unset += 1
            else:
                het.append(res.name)
        chains.append({"name": chain.name, "n_res": len(chain), "aa": aa, "het": het,
                       "unset": unset, "auth": auth})
    for i, c in enumerate(chains):
        report["aa_total"] += c["aa"]
        report["label_seq_unset"] += c["unset"]
        if c["het"] and any(later["aa"] for later in chains[i + 1:]):
            report["het_before_protein"].append((c["name"], sorted(set(c["het"]))))
        if c["auth"]:
            span = max(c["auth"]) - min(c["auth"]) + 1
            if span > c["aa"]:
                report["gaps"].append((c["name"], c["aa"], span))
    report["chains"] = chains
    return report

def prepare_aligned_model(src, dst, strip_ligands=True, label_seq_from="auto", verbose=True):
    """Sanitise any PDB/mmCIF into something CryoBoltz's guidance loader can index.

    Removes waters, hydrogens and altlocs, optionally drops ligands/metals (they have no place in the
    aligned model and, sitting inside a protein chain, they shift every later chain's residue index),
    then writes mmCIF *with* label_seq_id. `label_seq_from`: "auto" uses SEQRES when present and falls
    back to the author numbering for gapped models, "seqres" and "auth" force one or the other.
    """
    before = raw_structure_report(src)
    st = gemmi.read_structure(str(src))
    st.setup_entities()
    st.remove_waters()
    st.remove_hydrogens()
    st.remove_alternative_conformations()
    if strip_ligands:
        st.remove_ligands_and_waters()
    st.remove_empty_chains()
    st.assign_label_seq_id()

    gapped = [c for c in before["gaps"]]
    use_auth = label_seq_from == "auth" or (label_seq_from == "auto" and gapped)
    if use_auth:
        for model in st:
            for chain in model:
                for res in chain:
                    info = gemmi.find_tabulated_residue(res.name)
                    if info and info.is_amino_acid():
                        res.label_seq = res.seqid.num
    st.make_mmcif_document().write_file(str(dst))

    if verbose:
        after = raw_structure_report(dst)
        print(f"  {Path(src).name} → {Path(dst).name}")
        print(f"    residues with label_seq unset: {before['label_seq_unset']} → {after['label_seq_unset']}")
        dropped = sum(len(c["het"]) for c in before["chains"]) - sum(len(c["het"]) for c in after["chains"])
        if dropped:
            names = sorted({n for c in before["chains"] for n in c["het"]})
            print(f"    dropped {dropped} non-amino-acid residue(s): {', '.join(names)}")
        print(f"    label_seq taken from: {'author numbering' if use_auth else 'SEQRES/entity order'}")
        if gapped:
            for name, present, span in gapped:
                print(f"    ⚠️  chain {name} has gaps ({present} residues spanning {span} positions) — using the "
                      "author numbering assumes it matches your sequence file 1:1; check Step 4's identity row")
    return Path(dst)

def all_atoms(path, heavy_only=True):
    """(coords (N,3), atomic numbers (N,))."""
    xyz, znum = [], []
    for chain in read_structure(path)[0]:
        for res in chain:
            for atom in res:
                if heavy_only and atom.element == gemmi.Element("H"):
                    continue
                xyz.append([atom.pos.x, atom.pos.y, atom.pos.z])
                znum.append(atom.element.atomic_number)
    return np.array(xyz, dtype=np.float64).reshape(-1, 3), np.array(znum, dtype=np.float64)

def transform_structure(src, dst, rot, trans):
    """Write `src` with x → rot·x + trans applied, preserving label_seq_id."""
    st = gemmi.read_structure(str(src))
    st.setup_entities()
    tr = gemmi.Transform()
    tr.mat.fromlist([[float(v) for v in row] for row in np.asarray(rot)])
    tr.vec.fromlist([float(v) for v in np.asarray(trans)])
    for model in st:
        for chain in model:
            for res in chain:
                for atom in res:
                    atom.pos = gemmi.Position(*tr.apply(atom.pos))
    st.make_mmcif_document().write_file(str(dst))
    return Path(dst)

def ca_rmsd(path_a, path_b):
    """CA RMSD between two structures as they sit (no superposition)."""
    a, b = cif_chains(path_a), cif_chains(path_b)
    xs, ys = [], []
    for k in (k for k in a if k in b):
        n = min(len(a[k]["ca"]), len(b[k]["ca"]))
        xs.append(a[k]["ca"][:n])
        ys.append(b[k]["ca"][:n])
    if not xs:
        return float("nan")
    x, y = np.vstack(xs), np.vstack(ys)
    return float(np.sqrt(((x - y) ** 2).sum(1).mean()))

def synthesize_map(cif_path, out_path, resolution=5.0, spacing=1.5, pad=8.0):
    """Gaussian-splat a structure into an MRC map (CryoBoltz's own forward model)."""
    xyz, znum = all_atoms(cif_path)
    sigma = max(resolution / (np.sqrt(2.0) * np.pi), 0.75 * spacing)
    lo, hi = xyz.min(0) - pad, xyz.max(0) + pad
    nx, ny, nz = (int(v) for v in np.ceil((hi - lo) / spacing).astype(int) + 1)
    grid = np.zeros((nz, ny, nx), dtype=np.float32)
    offs = np.arange(-int(np.ceil(4 * sigma / spacing)), int(np.ceil(4 * sigma / spacing)) + 1)
    for (x, y, z), amp in zip(xyz, znum):
        cx, cy, cz = (np.array([x, y, z]) - lo) / spacing
        xs, ys, zs = int(round(cx)) + offs, int(round(cy)) + offs, int(round(cz)) + offs
        xs, ys, zs = xs[(xs >= 0) & (xs < nx)], ys[(ys >= 0) & (ys < ny)], zs[(zs >= 0) & (zs < nz)]
        if not (len(xs) and len(ys) and len(zs)):
            continue
        gx = np.exp(-(((xs - cx) * spacing) ** 2) / (2 * sigma**2))
        gy = np.exp(-(((ys - cy) * spacing) ** 2) / (2 * sigma**2))
        gz = np.exp(-(((zs - cz) * spacing) ** 2) / (2 * sigma**2))
        grid[np.ix_(zs, ys, xs)] += (amp * gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)
    grid /= max(float(grid.max()), 1e-8)
    return write_map(grid, out_path, spacing, lo)

def rigid_fit(cif_path, info, thresh=0.0, n_rotations=1500, n_refine=5, seed=0, verbose=True):
    """Rigid-body fit a structure into a map by maximising the mean map value at CA positions.

    A lightweight stand-in for ChimeraX `fitmap`: exhaustive random orientation search from the
    density centroid, then Powell refinement of the best few. Returns (rot, trans, stats) such that
    x → rot·x + trans places the input into the map.
    """
    from scipy.optimize import minimize
    from scipy.spatial.transform import Rotation

    ca = np.vstack([c["ca"] for c in cif_chains(cif_path).values() if len(c["ca"])])
    if not len(ca):
        raise ValueError("no CA atoms found in the input structure")
    com = ca.mean(0)
    centred = ca - com

    above = np.clip(info.data - thresh, 0, None)
    zz, yy, xx = np.nonzero(above > 0)
    if not len(xx):
        raise ValueError("no voxels above the threshold — lower it")
    w = above[zz, yy, xx].astype(np.float64)
    pts = np.stack([info.origin[0] + xx * info.voxel[0],
                    info.origin[1] + yy * info.voxel[1],
                    info.origin[2] + zz * info.voxel[2]], axis=1)
    centroid = (pts * w[:, None]).sum(0) / w.sum()

    def score(rotvec, trans):
        rot = Rotation.from_rotvec(rotvec).as_matrix()
        return float(info.sample(centred @ rot.T + trans).mean())

    rots = Rotation.random(n_rotations, random_state=seed)
    coords = np.einsum("rij,nj->rni", rots.as_matrix(), centred) + centroid
    coarse = info.sample(coords.reshape(-1, 3)).reshape(n_rotations, -1).mean(1)
    best = (-np.inf, None, None)
    for i in np.argsort(coarse)[::-1][:n_refine]:
        x0 = np.concatenate([rots[int(i)].as_rotvec(), centroid])
        res = minimize(lambda p: -score(p[:3], p[3:]), x0, method="Powell",
                       options={"maxiter": 4000, "xtol": 1e-2, "ftol": 1e-3})
        if -res.fun > best[0]:
            best = (-res.fun, res.x[:3].copy(), res.x[3:].copy())
    fit_score, rotvec, trans = best
    rot = Rotation.from_rotvec(rotvec).as_matrix()
    fitted = centred @ rot.T + trans
    stats = {"score": fit_score,
             "map_mean_above_thresh": float(info.data[info.data > thresh].mean()) if (info.data > thresh).any() else float("nan"),
             "ca_inside_density": float((info.sample(fitted) > thresh).mean())}
    if verbose:
        print(f"  mean map value at CA atoms : {stats['score']:.4f}")
        print(f"  map mean above threshold   : {stats['map_mean_above_thresh']:.4f}")
        print(f"  CA atoms inside density    : {100 * stats['ca_inside_density']:.1f}%")
    # rot·(x - com) + trans  ==  rot·x + (trans - rot·com)
    return rot, trans - rot @ com, stats

# ─────────────────────────────────────────────────────────── visualisation

TRACE_COLORS = ["#e4572e", "#17bebb", "#ffc914", "#76b041", "#a06cd5", "#2e86ab"]

def isosurface_figure(info, models, iso, max_grid=48, opacity=0.22, title=""):
    """Plotly figure: map isosurface + CA traces. `models` maps a label → structure path."""
    import plotly.graph_objects as go
    step = max(1, int(np.ceil(max(info.nx, info.ny, info.nz) / max_grid)))
    vol = np.ascontiguousarray(info.data[::step, ::step, ::step], dtype=np.float32)
    ax, ay, az = info.grid_axes()
    gz, gy, gx = np.meshgrid(az[::step], ay[::step], ax[::step], indexing="ij")
    fig = go.Figure(go.Isosurface(
        x=gx.ravel().astype(np.float32), y=gy.ravel().astype(np.float32), z=gz.ravel().astype(np.float32),
        value=vol.ravel(), isomin=float(iso), isomax=float(vol.max()), surface_count=1,
        colorscale=[[0, "#9fb8c8"], [1, "#9fb8c8"]], showscale=False, opacity=opacity,
        caps=dict(x_show=False, y_show=False, z_show=False), name="density", hoverinfo="skip"))
    for i, (label, path) in enumerate(models.items()):
        color = TRACE_COLORS[i % len(TRACE_COLORS)]
        for j, (cname, chain) in enumerate(cif_chains(path).items()):
            ca = chain["ca"]
            if not len(ca):
                continue
            fig.add_trace(go.Scatter3d(x=ca[:, 0], y=ca[:, 1], z=ca[:, 2], mode="lines",
                                       line=dict(color=color, width=5), name=f"{label} · {cname}",
                                       showlegend=(j == 0), legendgroup=label))
    fig.update_layout(title=title, height=620, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data", xaxis_title="x (Å)", yaxis_title="y (Å)", zaxis_title="z (Å)"),
                      legend=dict(orientation="h", y=-0.02))
    return fig

def show_table(rows, float_fmt="{:.4g}"):
    """Print a list of dicts as a plain aligned table (no pandas needed)."""
    rows = [{k: (float_fmt.format(v) if isinstance(v, float) else str(v)) for k, v in r.items()} for r in rows]
    if not rows:
        print("  (nothing)")
        return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(r[c]) for r in rows)) for c in cols}
    print("  " + " │ ".join(c.ljust(width[c]) for c in cols))
    print("  " + "─┼─".join("─" * width[c] for c in cols))
    for r in rows:
        print("  " + " │ ".join(r[c].ljust(width[c]) for c in cols))

print("✔ helpers loaded:", ", ".join(sorted(
    n for n in ("MapInfo", "bin_map", "ca_rmsd", "cif_chains", "crop_map", "download", "download_first",
                "estimate_pixel_size", "fix_origin", "flip_map", "gunzip", "isosurface_figure", "label_seq_span",
                "parse_sequence_file", "prepare_aligned_model", "raw_structure_report", "rcsb_fasta_to_boltz",
                "pick_threshold", "rigid_fit", "run_streaming", "set_voxel_size", "show_table", "suggest_threshold",
                "synthesize_map", "threshold_table", "transform_structure", "write_map",
                "write_protein_only_sequence"))))

## Step 2 · Model weights and cache

CryoBoltz uses the released Boltz-1 confidence checkpoint (`boltz1_conf.ckpt`, ≈2.2 GB) plus the CCD
dictionary (`ccd.pkl`, ≈60 MB). The CLI downloads them on first use; doing it here makes the progress
visible and lets you park the cache **on Google Drive** so a new Colab session doesn't re-download 2 GB.

In [ ]:
#@title Step 2 · Fetch weights (optionally cache them on Drive) { display-mode: "form" }
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
#@markdown Leave `CACHE_DIR` empty for the default: `<WORK_DIR>/cache`, or `MyDrive/cryoboltz_cache` with Drive on.
CACHE_DIR = "" #@param {type:"string"}

require("WORK_DIR")
from pathlib import Path

DRIVE_ROOT = None
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive")
    except Exception as exc:  # noqa: BLE001
        print(f"⚠️  could not mount Drive ({exc}) — falling back to local storage")

if CACHE_DIR.strip():
    CACHE_DIR = Path(CACHE_DIR.strip()).expanduser()
elif DRIVE_ROOT is not None:
    CACHE_DIR = DRIVE_ROOT / "cryoboltz_cache"
else:
    CACHE_DIR = Path(WORK_DIR) / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

try:
    from boltz.main import CCD_URL, MODEL_URL       # keeps the URLs in sync with the installed package
except Exception:                                    # noqa: BLE001
    CCD_URL = "https://huggingface.co/boltz-community/boltz-1/resolve/main/ccd.pkl"
    MODEL_URL = "https://huggingface.co/boltz-community/boltz-1/resolve/main/boltz1_conf.ckpt"

print(f"cache directory: {CACHE_DIR}\n")
download(CCD_URL, CACHE_DIR / "ccd.pkl")
download(MODEL_URL, CACHE_DIR / "boltz1_conf.ckpt")

sizes = {p.name: p.stat().st_size for p in CACHE_DIR.iterdir() if p.is_file()}
print("\ncontents:")
for name, size in sorted(sizes.items()):
    print(f"  {name:22s} {human(size)}")
if sizes.get("boltz1_conf.ckpt", 0) < 1_000_000_000:
    print("\n⚠️  The checkpoint looks too small — delete it and re-run this cell.")
else:
    print("\n✔ weights ready. Every `boltz predict` call below is given `--cache`, so nothing is re-downloaded.")

## Step 3 · Choose your inputs

Pick one of four modes:

| Mode | What it does | Needs |
|------|--------------|-------|
| **Smoke test** | 117-residue example from the repo, with a **synthetic** map built from an unguided prediction. Exercises the whole guidance pipeline end to end in ~10 min on a free T4. Start here to confirm your setup works. | GPU only |
| **Pma1 demo** | The paper's example: [PDB 9UGC](https://www.rcsb.org/structure/9UGC) + [EMD-64136](https://www.ebi.ac.uk/emdb/EMD-64136) at 3.52 Å, with the pre-aligned structure shipped in `examples/`. | ~40 GB GPU, ~1 GB download |
| **Upload** | Your own sequence, map and aligned structure from your computer. | — |
| **Fetch by ID / URL** | Pulls a sequence from RCSB, a map from EMDB, and an aligned model from a URL or local path. | internet |

Each mode ends by setting five variables the rest of the notebook uses: `SEQ_PATH`, `MAP_PATH`, `CIF_PATH`,
`RES` (estimated map resolution, Å) and `THRESH` (contour level below which the map is background).

In [ ]:
#@title Step 3 · Gather inputs { display-mode: "form" }
INPUT_MODE = "Smoke test — synthetic map, runs on a T4" #@param ["Smoke test — synthetic map, runs on a T4", "Pma1 demo — PDB 9UGC + EMD-64136 (needs ~40 GB GPU)", "Upload my own files", "Fetch by ID / URL"]
#@markdown ---
#@markdown **Fetch by ID / URL** settings (ignored by the other modes). `ALIGNED_MODEL` accepts a URL *or* a path already on this machine.
PDB_ID = "" #@param {type:"string"}
EMDB_ID = "" #@param {type:"string"}
SEQUENCE_URL = "" #@param {type:"string"}
MAP_URL = "" #@param {type:"string"}
ALIGNED_MODEL = "" #@param {type:"string"}
KEEP_N_CHAINS = 0 #@param {type:"integer"}
#@markdown ---
#@markdown Set these if you know them. `0` means "look it up / suggest it for me".
RESOLUTION = 0 #@param {type:"number"}
MAP_THRESHOLD = 0 #@param {type:"number"}

require("WORK_DIR", "REPO_DIR", "CACHE_DIR", "BOLTZ_CMD")
import json, urllib.request
from pathlib import Path

INPUT_DIR = Path(WORK_DIR) / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
SEQ_PATH = MAP_PATH = CIF_PATH = None
RES, THRESH = float(RESOLUTION), float(MAP_THRESHOLD)

EMDB_MIRRORS = [
    "https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-{i}/map/emd_{i}.map.gz",
    "https://files.wwpdb.org/pub/emdb/structures/EMD-{i}/map/emd_{i}.map.gz",
    "https://ftp.rcsb.org/pub/emdb/structures/EMD-{i}/map/emd_{i}.map.gz",
    "https://ftp.pdbj.org/pub/emdb/structures/EMD-{i}/map/emd_{i}.map.gz",
]
RCSB_FASTA = ["https://www.rcsb.org/fasta/entry/{i}/download", "https://www.rcsb.org/fasta/entry/{i}"]

def emdb_metadata(emdb_id):
    """(resolution, recommended contour level) from the EMDB API — best effort."""
    res = contour = None
    try:
        with urllib.request.urlopen(f"https://www.ebi.ac.uk/emdb/api/entry/EMD-{emdb_id}", timeout=60) as fh:
            doc = json.load(fh)
    except Exception as exc:  # noqa: BLE001
        print(f"  (could not reach the EMDB API: {exc})")
        return None, None
    def walk(node):
        nonlocal res, contour
        if isinstance(node, dict):
            if res is None and isinstance(node.get("resolution"), dict):
                val = node["resolution"].get("valueOf_", node["resolution"].get("value"))
                if val is not None:
                    try:
                        res = float(val)
                    except (TypeError, ValueError):
                        pass
            if contour is None and node.get("contour") is not None:
                c = node["contour"]
                c = c[0] if isinstance(c, list) and c else c
                if isinstance(c, dict) and c.get("level") is not None:
                    try:
                        contour = float(c["level"])
                    except (TypeError, ValueError):
                        pass
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for v in node:
                walk(v)
    walk(doc)
    return res, contour

def fetch_or_copy(spec, dst):
    """Accept an http(s) URL or an existing local path."""
    spec = str(spec).strip()
    if spec.lower().startswith(("http://", "https://", "ftp://")):
        path = download(spec, dst)
    else:
        src = Path(spec).expanduser()
        if not src.exists():
            raise FileNotFoundError(f"{src} does not exist")
        path = Path(dst)
        if src.resolve() != path.resolve():
            path.write_bytes(src.read_bytes())
        print(f"  ✔ using local {src.name} ({human(path.stat().st_size)})")
    return gunzip(path) if path.suffix == ".gz" else path

# ───────────────────────────────────────────────── mode 1: smoke test
if INPUT_MODE.startswith("Smoke test"):
    SEQ_PATH = (Path(REPO_DIR) / "examples/prot.fasta").resolve()
    smoke = Path(WORK_DIR) / "smoke"
    unguided = smoke / "unguided"
    model0 = unguided / "boltz_results_prot/predictions/prot/prot_model_0.cif"
    print("The smoke test needs a structure to build a synthetic map from, so it first runs a short")
    print("unguided Boltz prediction on examples/prot.fasta (117 residues, MSA already in the repo).\n")
    if model0.exists():
        print(f"✔ reusing the unguided prediction at {model0}")
    else:
        rc = run_streaming([*BOLTZ_CMD, "predict", str(SEQ_PATH), "--out_dir", str(unguided),
                            "--cache", str(CACHE_DIR), "--diffusion_samples", "1",
                            "--sampling_steps", "100", "--output_format", "mmcif",
                            "--num_workers", "1", "--override"],
                           cwd=REPO_DIR, log_path=Path(WORK_DIR) / "logs/smoke_unguided.log")
        if rc != 0 or not model0.exists():
            raise RuntimeError("the unguided warm-up prediction failed — see the log above")
    CIF_PATH = model0.resolve()
    RES = 5.0
    MAP_PATH = synthesize_map(CIF_PATH, INPUT_DIR / "smoke_map.mrc", resolution=RES, spacing=1.5, pad=8.0)
    info = MapInfo(MAP_PATH)
    n_res = sum(len(s) for _, t, s in parse_sequence_file(SEQ_PATH) if t == "protein")
    THRESH = round(suggest_threshold(info, expected_volume(n_res)), 4)
    print(f"\n✔ synthetic {RES:.1f} Å map: {info.nx}×{info.ny}×{info.nz} voxels at 1.5 Å")
    print("  (built from the unguided prediction, so a working guided run should reproduce it closely)")

# ───────────────────────────────────────────────── mode 2: Pma1 from the paper
elif INPUT_MODE.startswith("Pma1"):
    print("Sequence — PDB 9UGC (keeping one chain, to model the monomer):")
    raw = INPUT_DIR / "9UGC_rcsb.fasta"
    download_first([u.format(i="9UGC") for u in RCSB_FASTA], raw)
    SEQ_PATH = INPUT_DIR / "pma1.fasta"
    print("  chains written:", rcsb_fasta_to_boltz(raw.read_text(), SEQ_PATH, keep_n_chains=1))
    print("\nMap — EMD-64136 (~hundreds of MB, be patient):")
    gz = download_first([u.format(i="64136") for u in EMDB_MIRRORS], INPUT_DIR / "emd_64136.map.gz")
    MAP_PATH = gunzip(gz, INPUT_DIR / "emd_64136.map")
    CIF_PATH = (Path(REPO_DIR) / "examples/input_struct.cif").resolve()
    print(f"\nAligned model — {CIF_PATH.name} from the repo (an unguided Boltz prediction fitted to this map)")
    RES = RES or 3.52          # reported resolution
    THRESH = THRESH or 0.35    # chosen by eye in ChimeraX, per the README
    print("\n⚠️  918 residues: expect an A100/L4 and a long run. Crop the map in Step 5 (it matters a lot).")

# ───────────────────────────────────────────────── mode 3: upload
elif INPUT_MODE.startswith("Upload"):
    from google.colab import files
    print("Select your sequence (.fasta/.yaml), map (.mrc/.map/.ccp4, optionally .gz) and aligned model (.cif):")
    for name, blob in files.upload().items():
        dst = INPUT_DIR / name
        dst.write_bytes(blob)
        low = name.lower()
        if low.endswith(".gz"):
            dst = gunzip(dst)
            low = dst.name.lower()
        if low.endswith((".fasta", ".fa", ".fas", ".yaml", ".yml")):
            SEQ_PATH = dst
        elif low.endswith((".mrc", ".map", ".ccp4", ".mrcs")):
            MAP_PATH = dst
        elif low.endswith((".cif", ".mmcif", ".pdb")):
            CIF_PATH = dst
        print(f"  ✔ {dst.name} ({human(dst.stat().st_size)})")

# ───────────────────────────────────────────────── mode 4: fetch by id / url
else:
    if SEQUENCE_URL.strip():
        SEQ_PATH = fetch_or_copy(SEQUENCE_URL, INPUT_DIR / Path(SEQUENCE_URL.split("?")[0]).name)
    elif PDB_ID.strip():
        pdb = PDB_ID.strip().upper()
        raw = INPUT_DIR / f"{pdb}_rcsb.fasta"
        download_first([u.format(i=pdb) for u in RCSB_FASTA], raw)
        SEQ_PATH = INPUT_DIR / f"{pdb.lower()}.fasta"
        mapping = rcsb_fasta_to_boltz(raw.read_text(), SEQ_PATH, keep_n_chains=KEEP_N_CHAINS)
        print("  chains written:", mapping)
        print("  ⚠️  check these labels match the chains of your aligned model (Step 4 verifies it).")
    else:
        raise ValueError("give either SEQUENCE_URL or PDB_ID")

    if MAP_URL.strip():
        MAP_PATH = fetch_or_copy(MAP_URL, INPUT_DIR / Path(MAP_URL.split("?")[0]).name)
    elif EMDB_ID.strip():
        emd = EMDB_ID.strip().upper().replace("EMD-", "")
        gz = download_first([u.format(i=emd) for u in EMDB_MIRRORS], INPUT_DIR / f"emd_{emd}.map.gz")
        MAP_PATH = gunzip(gz, INPUT_DIR / f"emd_{emd}.map")
        meta_res, meta_contour = emdb_metadata(emd)
        if meta_res:
            print(f"  EMDB reports a resolution of {meta_res:.2f} Å")
            RES = RES or meta_res
        if meta_contour is not None:
            print(f"  EMDB recommends contour level {meta_contour:g}")
            THRESH = THRESH or meta_contour
    else:
        raise ValueError("give either MAP_URL or EMDB_ID")

    if ALIGNED_MODEL.strip():
        CIF_PATH = fetch_or_copy(ALIGNED_MODEL, INPUT_DIR / Path(ALIGNED_MODEL.split("?")[0]).name)
    else:
        print("\nNo aligned model given — Step 6 can build one (unguided prediction + rigid-body fit).")

MAP_PATH_RAW = MAP_PATH
if not RES:
    RES = 3.0
    print(f"\n⚠️  No resolution given; defaulting to {RES} Å. Set RESOLUTION — the local guidance term uses it.")
print("\n" + "─" * 78)
print(f"SEQ_PATH  = {SEQ_PATH}")
print(f"MAP_PATH  = {MAP_PATH}")
print(f"CIF_PATH  = {CIF_PATH}")
print(f"RES       = {RES} Å        THRESH = {THRESH}" + ("   (0 → Step 4 will suggest one)" if not THRESH else ""))
USE_MSA_SERVER = not has_precomputed_msa(SEQ_PATH) if SEQ_PATH else True
print(f"MSA       = {'ColabFold MMseqs2 server (--use_msa_server)' if USE_MSA_SERVER else 'precomputed, from the input file'}")

### Step 3b · *cryoDRGN volumes* (optional)

Run this if your maps come out of **cryoDRGN** (`eval_vol`, `analyze`, `abinit_het`, cryoDRGN-AI …). Its
conventions suit CryoBoltz well — cubic boxes, isotropic voxels, `mapc/mapr/maps = 1,2,3`, `nstart = 0`, and
the header `origin` is the only offset (set to `(D_old−D_crop)/2·Apix` when `eval_vol --crop` is used), which
is exactly the field CryoBoltz reads. Two things do need attention:

1. **Pixel size.** `cryodrgn eval_vol --Apix` **defaults to 1.0**, and `cryodrgn analyze` falls back to 1.0
   when it cannot read a unique pixel size from the CTF parameters. A volume that claims 1 Å/px when it is
   really, say, 1.63 Å/px is geometrically wrong, and `--res` then means nothing. If you already aligned an
   atomic model into the volume and it fits, your header is probably right — this cell measures that
   independently by comparing the radius of gyration of the density with that of your model.
2. **Many volumes, one run each.** cryoDRGN gives you an *ensemble* (k-means centres, PC traversals). CryoBoltz
   fits one map at a time, so this cell collects the set and **Step 7b** runs them in a loop, reusing the MSA.

If your model is already aligned to these volumes (in ChimeraX, say), you are in good shape: ChimeraX places
voxel *(i,j,k)* at `origin + (i,j,k)·Apix`, the same as CryoBoltz, so the alignment carries over untouched.
Preparing the model in Step 6 does not move any atom — it only removes residues and rewrites the file — so run
it if Step 4 complains, and keep `RIGID_FIT_TO_MAP` **off**.

In [ ]:
#@title Step 3b · Collect cryoDRGN volumes and check their pixel size { display-mode: "form" }
#@markdown Directory, single file, or glob — e.g. `/content/drive/MyDrive/…/analyze.49/kmeans20/vol_*.mrc`
CRYODRGN_VOLUMES = "" #@param {type:"string"}
#@markdown `0` keeps whatever the header says. Set the true value if cryoDRGN wrote the default 1.0.
CRYODRGN_APIX = 0 #@param {type:"number"}
#@markdown Which volume of the set to use for Steps 4–7 (Step 7b runs all of them).
PRIMARY_VOLUME_INDEX = 0 #@param {type:"integer"}
CHECK_PIXEL_SIZE_AGAINST_MODEL = True #@param {type:"boolean"}

require("WORK_DIR", "INPUT_DIR")
import glob
from pathlib import Path

spec = CRYODRGN_VOLUMES.strip()
if not spec:
    raise ValueError("point CRYODRGN_VOLUMES at a directory, a file, or a glob of .mrc volumes")
p = Path(spec).expanduser()
if p.is_dir():
    files = sorted(p.glob("*.mrc")) + sorted(p.glob("*.map"))
elif any(ch in spec for ch in "*?["):
    files = sorted(Path(f) for f in glob.glob(str(Path(spec).expanduser())))
else:
    files = [p]
files = [f for f in files if f.is_file()]
if not files:
    raise FileNotFoundError(f"no volumes matched {spec}")

if CRYODRGN_APIX > 0:
    fixed_dir = Path(INPUT_DIR) / "cryodrgn_apix"
    fixed_dir.mkdir(parents=True, exist_ok=True)
    print(f"■ rewriting the header pixel size to {CRYODRGN_APIX} Å/px (originals untouched)")
    files = [set_voxel_size(f, fixed_dir / f.name, CRYODRGN_APIX) for f in files]

rows, infos = [], []
for f in files:
    info = MapInfo(f)
    infos.append(info)
    rows.append({"#": len(rows), "file": f.name, "box": f"{info.nx}³" if info.nx == info.ny == info.nz
                 else f"{info.nx}×{info.ny}×{info.nz}", "Å/px": round(info.spacing, 4),
                 "origin": "(0,0,0)" if np.allclose(info.origin, 0) else
                           "(" + ", ".join(f"{v:.1f}" for v in info.origin) + ")",
                 "min": round(float(info.data.min()), 4), "max": round(float(info.data.max()), 4)})
print(f"\n■ {len(files)} volume(s)")
show_table(rows)

spacings = {round(i.spacing, 4) for i in infos}
if len(spacings) > 1:
    print(f"\n⚠️  the set does not share one pixel size ({sorted(spacings)}) — check what produced these files")
if any(abs(i.spacing - 1.0) < 1e-6 for i in infos) and CRYODRGN_APIX <= 0:
    print("\n⚠️  Header says exactly 1.0 Å/px. That is cryoDRGN's default when `--Apix` is not given, so it may "
          "not be your real pixel size. If it is wrong, set CRYODRGN_APIX (the true value is\n"
          "    `original_Apix × original_box / cryodrgn_box`, and `cryodrgn analyze` prints it as `using A/px=…`).")

VOLUME_LIST = [Path(f) for f in files]
idx = max(0, min(PRIMARY_VOLUME_INDEX, len(VOLUME_LIST) - 1))
MAP_PATH = MAP_PATH_RAW = VOLUME_LIST[idx]
print(f"\n✔ VOLUME_LIST holds {len(VOLUME_LIST)} volume(s); primary = [{idx}] {MAP_PATH.name}")

info = infos[idx]
if not THRESH:
    if globals().get("SEQ_PATH"):
        n_res = sum(len(s) for _, t, s in parse_sequence_file(SEQ_PATH) if t == "protein")
        THRESH = pick_threshold(info, expected_volume(n_res))
        print(f"  THRESH ← {THRESH:g} (level enclosing the expected molecular volume of {n_res} residues)")
    else:
        THRESH = float(f"{float(np.percentile(info.data, 99.5)):.4g}")
        print(f"  THRESH ← {THRESH:g} (99.5th percentile; set SEQ_PATH in Step 3 for a better estimate)")

if CHECK_PIXEL_SIZE_AGAINST_MODEL and globals().get("CIF_PATH"):
    est = estimate_pixel_size(info, CIF_PATH, THRESH)
    print(f"\n■ pixel-size cross-check against {Path(CIF_PATH).name}")
    print(f"  radius of gyration — density {est['rg_map']:.1f} Å · model {est['rg_model']:.1f} Å")
    print(f"  implied scale {est['scale']:.3f} → implied spacing {est['implied_spacing']:.4f} Å/px "
          f"(header says {info.spacing:.4f})")
    if not np.isfinite(est["scale"]):
        print("  ⚠️  could not measure the density's Rg — is the threshold sensible?")
    elif abs(est["scale"] - 1.0) > 0.10:
        print("  ⚠️  more than 10% off. Either the header pixel size is wrong (try CRYODRGN_APIX = "
              f"{est['implied_spacing']:.3f}), or the map holds more/less matter than your model "
              "(extra subunits, a partial model, or a threshold that is too loose).")
    else:
        print("  ✔ consistent within 10% — the header pixel size looks right for this model")

if globals().get("CIF_PATH"):
    ca = np.vstack([v["ca"] for v in cif_chains(CIF_PATH).values() if len(v["ca"])])
    lo, hi = info.box
    in_box = float(((ca >= lo) & (ca <= hi)).all(1).mean())
    in_density = float((info.sample(ca) > THRESH).mean())
    print(f"\n■ is the model already in this volume's frame?")
    print(f"  CA atoms inside the box     : {100 * in_box:.1f}%")
    print(f"  CA atoms in density > {THRESH:g}: {100 * in_density:.1f}%")
    if in_box > 0.95 and in_density > 0.7:
        print("  ✔ your existing alignment transfers to CryoBoltz as-is — leave RIGID_FIT_TO_MAP off in Step 6.")
    elif in_box < 0.9:
        print("  ❌ the model largely falls outside this volume's box. Either it was aligned to a different "
              "volume/frame, or the header pixel size is wrong (see the cross-check above). Fix the pixel size "
              "first; only then consider RIGID_FIT_TO_MAP in Step 6.")
    else:
        print("  ⚠️  the model is in the box but mostly outside the density. Check the threshold, then look at the "
              "overlay (Step 6 with SHOW_OVERLAY, or Step 9b) before spending GPU time.")
print("\nNext: Step 4 to validate, Step 5 to crop (cryoDRGN boxes are mostly background), then Step 7 or 7b.")

## Step 4 · Validate the inputs

Most CryoBoltz failures are input problems, not modelling problems, and they are cheap to catch here
instead of 40 minutes into a run. This cell checks:

* map header sanity — cubic voxels, axis order, and the classic **`nstart` vs `origin`** trap (CryoBoltz
  reads only `origin`, so a map that stores its offset in `nstart` is silently shifted);
* whether the aligned model actually sits **inside** the map box;
* that chain labels and residue identities in the aligned model **match the sequence file** — parsed with
  CryoBoltz's own `load_cif`, so what you see here is exactly what guidance will see;
* a contour **threshold suggestion** from the enclosed volume;
* how big the run will be, and a `--voxel_batch` that should fit your GPU.

In [ ]:
#@title Step 4 · Validate sequence ↔ structure ↔ map { display-mode: "form" }
SHOW_HISTOGRAM = True #@param {type:"boolean"}

require("SEQ_PATH", "MAP_PATH", "RES")
import io, contextlib
import numpy as np

problems, warnings_ = [], []

# ── sequence ────────────────────────────────────────────────────────────
records = parse_sequence_file(SEQ_PATH)
print(f"■ sequence file: {SEQ_PATH}")
show_table([{"chain": c, "type": t, "length": len(s)} for c, t, s in records])
prot = [(c, s) for c, t, s in records if t == "protein"]
n_res = sum(len(s) for _, s in prot)
if not prot:
    problems.append("no protein chains found — CryoBoltz guidance only models protein atoms")
non_protein = [(c, t) for c, t, _ in records if t != "protein"]
if non_protein:
    problems.append(f"the sequence file contains non-protein entities ({', '.join(f'{c}:{t}' for c, t in non_protein)}"
                    "). Guidance's forward model is protein-only: it builds scattering amplitudes from per-residue "
                    "templates and gives every ligand atom the 5-atom UNK template, so the local term's amplitude "
                    "vector stops matching the real atom count. Predict ligands with plain Boltz-1 if you need them; "
                    "guide on the protein-only system — Step 6's DROP_NON_PROTEIN_FROM_SEQUENCE writes that "
                    "file for you")

# exact atom count CryoBoltz will guide, straight from the package tables
try:
    from boltz.data import const
    N_ATOMS = sum(len(const.ref_atoms[const.prot_letter_to_token[aa]]) for _, s in prot for aa in s)
except Exception as exc:  # noqa: BLE001
    N_ATOMS = int(7.9 * n_res)
    warnings_.append(f"could not use boltz tables for the atom count ({exc}); estimating {N_ATOMS}")
print(f"  → {len(prot)} protein chain(s), {n_res} residues, {N_ATOMS} guided atoms")

# ── map ─────────────────────────────────────────────────────────────────
info = MapInfo(MAP_PATH)
print(f"\n■ density map: {MAP_PATH}")
print("   ", info.summary())
for msg in info.issues():
    problems.append(f"map: {msg}")

target_volume = expected_volume(n_res)
print(f"\n■ contour threshold  (expected molecular volume ≈ {target_volume:,.0f} Å³)")
show_table(threshold_table(info, target_volume))
suggested = pick_threshold(info, target_volume)
print(f"  suggested threshold ≈ {suggested:g}  (encloses about the expected volume)")
if not THRESH:
    THRESH = suggested
    print(f"  → THRESH was 0, using the suggestion: {THRESH:g}")
else:
    enclosed = float((info.data > THRESH).sum()) * info.voxel_volume
    print(f"  your THRESH = {THRESH:g} encloses {enclosed:,.0f} Å³ "
          f"({enclosed / target_volume:.2f}× the expected volume)")
    if enclosed / target_volume > 4:
        warnings_.append("your threshold encloses far more volume than the molecule — background noise will "
                         "drag the global guidance term; consider raising it")
    if enclosed / target_volume < 0.25:
        warnings_.append("your threshold encloses much less volume than the molecule — parts of the map will "
                         "be treated as background; consider lowering it")

# ── aligned model ───────────────────────────────────────────────────────
if CIF_PATH:
    chains = cif_chains(CIF_PATH)
    print(f"\n■ aligned model: {CIF_PATH}")
    show_table([{"chain": k, "residues": v["n_res"], "CA atoms": len(v["ca"]),
                 "auth numbering": f"{min(v['auth_seq'])}–{max(v['auth_seq'])}",
                 "label_seq": label_seq_span(v)} for k, v in chains.items()])

    # read it the way load_cif does (no entity setup) — that is what guidance will actually see
    raw = raw_structure_report(CIF_PATH)
    if raw["label_seq_unset"]:
        problems.append(f"model: {raw['label_seq_unset']} residue(s) have no label_seq_id. CryoBoltz indexes the "
                        "aligned model by label_seq_id, and gemmi leaves it unset for PDB files, so the run dies "
                        "with `TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'`. Turn on "
                        "PREPARE_ALIGNED_MODEL in Step 6 (or convert yourself: `gemmi convert --to=mmcif`)")
    for name, het in raw["het_before_protein"]:
        problems.append(f"model: chain {name} contains non-amino-acid residue(s) ({', '.join(het)}) and is followed "
                        "by more protein chains. CryoBoltz offsets each chain by the total residue count of the "
                        "previous chain, so those extra residues shift every later chain (IndexError, or silent "
                        "mismatches). Strip ligands/metals from the aligned model — Step 6 can do it")
    for name, present, span in raw["gaps"]:
        warnings_.append(f"model: chain {name} has {present} residues spanning {span} positions, i.e. gaps. Residue "
                         "indices must line up with your sequence file — check the identity row below, and see "
                         "PREPARE_ALIGNED_MODEL in Step 6 if they do not")

    ca_all = np.vstack([v["ca"] for v in chains.values() if len(v["ca"])])
    lo, hi = info.box
    inside = float(((ca_all >= lo) & (ca_all <= hi)).all(1).mean())
    print(f"  model bbox   : {np.round(ca_all.min(0), 1)} → {np.round(ca_all.max(0), 1)} Å")
    print(f"  map box      : {np.round(lo, 1)} → {np.round(hi, 1)} Å")
    print(f"  CA inside box: {100 * inside:.1f}%")
    if inside < 0.9:
        problems.append(f"model: only {100 * inside:.0f}% of CA atoms fall inside the map box — the structure "
                        "is not aligned to this map (or the map origin is wrong). Use Step 6 to fit it")
    else:
        occupied = float((info.sample(ca_all) > THRESH).mean())
        print(f"  CA in density (> {THRESH:g}): {100 * occupied:.1f}%")
        if occupied < 0.5:
            warnings_.append(f"only {100 * occupied:.0f}% of CA atoms sit in density above the threshold — the "
                             "alignment is probably poor; check the overlay in Step 6/9")

    # chain-by-chain comparison against the sequence file
    print("\n  sequence ↔ model agreement:")
    rows = []
    for (cid, seq), (mid, mchain) in zip(prot, chains.items()):
        n = min(len(seq), len(mchain["seq"]))
        ident = sum(a == b for a, b in zip(seq[:n], mchain["seq"][:n])) / max(n, 1)
        rows.append({"seq chain": cid, "model chain": mid, "seq len": len(seq),
                     "model res": mchain["n_res"], "identity": f"{100 * ident:.1f}%"})
        if cid != mid:
            warnings_.append(f"chain label mismatch: sequence '{cid}' vs model '{mid}' — CryoBoltz matches "
                             "chains by order and label; rename them to agree")
        if ident < 0.95:
            problems.append(f"chain {mid}: only {100 * ident:.0f}% of residues match the sequence — wrong chain, "
                            "wrong order or an offset in numbering")
    show_table(rows)
    if len(prot) > 1 and any(len(s) != chains[m]["n_res"] for (_, s), m in zip(prot[:-1], list(chains)[:-1])):
        warnings_.append("multi-chain input where a non-final model chain has missing residues: CryoBoltz "
                         "offsets chains by the number of residues present in the file, so later chains can "
                         "shift. Fill the gaps or model one chain at a time")

    # …and the real thing: CryoBoltz's own loader
    try:
        from boltz.model.modules.struct_tools import load_cif as _load_cif
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            coords, mask = _load_cif(str(CIF_PATH), "".join(s for _, s in prot), ca_only=True)
        n_matched, n_warn = int(mask.sum()), buf.getvalue().count("WARNING")
        print(f"\n  CryoBoltz load_cif(): matched {n_matched}/{n_res} CA atoms" +
              (f", {n_warn} residue mismatch warning(s)" if n_warn else ""))
        for line in [l for l in buf.getvalue().splitlines() if l.strip()][:5]:
            print("    ", " ".join(line.split()))
        if n_matched < 0.5 * n_res:
            problems.append(f"CryoBoltz could only match {n_matched} of {n_res} CA atoms in the aligned model — "
                            "guidance would be initialised from almost nothing")
        elif n_matched < 0.9 * n_res:
            warnings_.append(f"CryoBoltz matched only {n_matched}/{n_res} CA atoms (missing residues are fine, "
                             "but check the coverage is what you expect)")
    except ImportError as exc:
        warnings_.append(f"boltz is not importable here, so its loader could not be run ({exc}) — restart the "
                         "session and re-run Steps 1 and 1b")
    except Exception as exc:  # noqa: BLE001
        problems.append(f"CryoBoltz's own load_cif() raised {type(exc).__name__}: {exc}. This is exactly what the "
                        "guided run will do when it initialises guidance — fix the aligned model (Step 6) before "
                        "spending GPU time")
else:
    problems.append("no aligned model yet — run Step 6, or set CIF_PATH")

# ── run size ────────────────────────────────────────────────────────────
print("\n■ run size")
suggested_batch = int(2 ** int(np.floor(np.log2(max(1.5e9 / (12 * max(N_ATOMS, 1)), 1024)))))
SUGGESTED_VOXEL_BATCH = int(min(32768, max(1024, suggested_batch)))
cloud_points = int(0.25 * N_ATOMS // max(info.spacing, 1e-6) ** 3)
cloud_points = min(cloud_points, int((info.data > THRESH).sum()))
print(f"  voxels in map            : {info.n_voxels:,}")
print(f"  local-guidance batches   : {int(np.ceil(info.n_voxels / SUGGESTED_VOXEL_BATCH)):,} per sample per step "
      f"(steps 151–175 by default)")
print(f"  suggested --voxel_batch  : {SUGGESTED_VOXEL_BATCH}  (keeps the {N_ATOMS}×batch distance tensor ≈1.5 GB)")
print(f"  global-guidance cloud    : ≈{cloud_points:,} points at --cloud_size 0.25")
if N_ATOMS * max(cloud_points, 1) > 25_000_000:
    warnings_.append(f"the global guidance point cloud ({cloud_points:,} points × {N_ATOMS} atoms) exceeds "
                     "geomloss's tensorized limit (5000²), so it will switch to the PyKeOps backend and try to "
                     "compile CUDA kernels — usually slow and sometimes broken on Colab. Bin the map in Step 5 "
                     "or lower --cloud_size in Step 7")
if info.n_voxels > 20_000_000:
    warnings_.append(f"{info.n_voxels:,} voxels is a lot for local guidance — crop and/or bin in Step 5")

# ── histogram ───────────────────────────────────────────────────────────
if SHOW_HISTOGRAM:
    import matplotlib.pyplot as plt
    sample = info.data.ravel()
    if sample.size > 4_000_000:
        sample = sample[:: sample.size // 4_000_000]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.hist(sample, bins=200, color="#7f9db9", log=True)
    ax.axvline(THRESH, color="#e4572e", lw=2, label=f"THRESH = {THRESH:g}")
    ax.axvline(suggested, color="#2e8b57", lw=2, ls="--", label=f"suggested = {suggested:.4g}")
    ax.set(xlabel="map value", ylabel="voxel count (log)", title="Map value distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

# ── verdict ─────────────────────────────────────────────────────────────
print("\n" + "═" * 78)
for msg in warnings_:
    print(f"⚠️  {msg}")
for msg in problems:
    print(f"❌ {msg}")
if not problems:
    print("✅ inputs look consistent — continue to Step 5 (crop the map) or jump to Step 7 (run).")
else:
    print(f"\n{len(problems)} problem(s) above will very likely make the run fail or produce garbage. Fix them first.")

## Step 5 · Crop and bin the map  *(recommended)*

Local guidance walks **every voxel** of the map on the guided diffusion steps, so background voxels cost
real time and memory. Two levers:

* **Crop** to a tight box around the density (this is `scripts/preproc_map.py` from the repo, the same command
  the README uses).
* **Bin** (block-average) the map when its sampling is much finer than its resolution. A Nyquist-ish spacing of
  ~res/3 keeps all the information; binning by 2 removes 7/8 of the voxels.

Cropping and binning both rewrite the header origin correctly, so the map stays in the same physical frame.

In [ ]:
#@title Step 5 · Crop / bin { display-mode: "form" }
CROP_MODE = "threshold + padding (recommended)" #@param ["threshold + padding (recommended)", "fixed cube", "none"]
PAD_VOXELS = 10 #@param {type:"integer"}
CUBE_DIM = 128 #@param {type:"integer"}
#@markdown Leave `CROP_THRESHOLD` at 0 to crop at `THRESH` from Step 4.
CROP_THRESHOLD = 0 #@param {type:"number"}
#@markdown Mirror the map along one axis — the fix for a reconstruction of the wrong handedness.
#@markdown `z` reproduces `cryodrgn eval_vol --flip` exactly (it flips axis 0 of a `[z,y,x]` array).
#@markdown **A flip moves the density to the other side of the box, so any existing alignment is void:**
#@markdown re-fit with `RIGID_FIT_TO_MAP` in Step 6 afterwards. Not sure whether you need it? Run the
#@markdown handedness test in Step 6 first — it costs a second and tells you which hand fits better.
FLIP_AXIS = "none" #@param ["none", "z", "y", "x"]
#@markdown ---
#@markdown Fold `nstart` into the header origin (see Step 4's warning) before cropping.
FIX_ORIGIN = True #@param {type:"boolean"}
#@markdown Block-average by this factor (1 = off). Only sensible while the spacing stays ≲ resolution/3.
BIN_FACTOR = 1 #@param {type:"slider", min:1, max:4, step:1}

require("MAP_PATH_RAW", "THRESH", "RES", "INPUT_DIR", "REPO_DIR")
import subprocess, sys
from pathlib import Path

src = Path(MAP_PATH_RAW)
before = MapInfo(src)
print(f"■ input map: {src.name}")
print("   ", before.summary(), "\n")
work = src

if FLIP_AXIS != "none":
    flipped = Path(INPUT_DIR) / f"{src.stem}_{FLIP_AXIS}flip.mrc"
    work = flip_map(work, flipped, axis=FLIP_AXIS)
    print(f"■ mirrored along {FLIP_AXIS} → {flipped.name}  (same box, density on the other side)")
    print("  ⚠️  any alignment made against the unflipped map is now invalid — re-fit in Step 6\n")

if FIX_ORIGIN and np.allclose(before.origin, 0) and np.any(before.nstart != 0):
    fixed = Path(INPUT_DIR) / (src.stem + "_origin.mrc")
    print(f"■ folding nstart into origin → {fix_origin(work, fixed)}")
    work = fixed

if BIN_FACTOR > 1:
    new_spacing = before.spacing * BIN_FACTOR
    if new_spacing > RES / 2.0:
        print(f"⚠️  binning by {BIN_FACTOR} gives {new_spacing:.2f} Å voxels for a {RES} Å map — that is below "
              "Nyquist (res/2), so real signal is lost. Use a smaller factor.")
    elif new_spacing > RES / 3.0:
        print(f"ℹ️  binning by {BIN_FACTOR} gives {new_spacing:.2f} Å voxels (between res/3 and res/2) — fine for "
              "speed, slightly coarser than ideal for the local guidance term.")
    binned = Path(INPUT_DIR) / (src.stem + f"_bin{BIN_FACTOR}.mrc")
    work = bin_map(work, binned, BIN_FACTOR)
    print(f"■ binned by {BIN_FACTOR} → {MapInfo(work).summary()}\n")

if CROP_MODE != "none":
    cropped = Path(INPUT_DIR) / (src.stem + "_cropped.mrc")
    if CROP_MODE.startswith("threshold"):
        thresh = CROP_THRESHOLD or THRESH
        print(f"■ cropping to the density above {thresh:g} with {PAD_VOXELS} voxels of padding")
        work = crop_map(work, cropped, REPO_DIR, thresh=thresh, pad=PAD_VOXELS)
    else:
        print(f"■ cropping to a centred {CUBE_DIM}³ cube")
        work = crop_map(work, cropped, REPO_DIR, dim=CUBE_DIM)

MAP_PATH = Path(work)
after = MapInfo(MAP_PATH)
print(f"\n■ map for the run: {MAP_PATH.name}")
print("   ", after.summary())
shrink = before.n_voxels / max(after.n_voxels, 1)
print(f"\n✔ {before.n_voxels:,} → {after.n_voxels:,} voxels ({shrink:.1f}× fewer; local guidance scales with this)")
if CIF_PATH:
    ca = np.vstack([v["ca"] for v in cif_chains(CIF_PATH).values() if len(v["ca"])])
    lo, hi = after.box
    kept = float(((ca >= lo) & (ca <= hi)).all(1).mean())
    print(f"  aligned model still inside the box: {100 * kept:.1f}% of CA atoms"
          + ("" if kept > 0.98 else "   ⚠️  cropped too tightly — raise PAD_VOXELS or lower CROP_THRESHOLD"))

## Step 6 · *Optional:* build and align the initial structure

CryoBoltz needs a structure that is **roughly in the map's frame** (it is used once, to rigid-align the
diffusion trajectory to the map; it is never used as a template). The README suggests an unguided Boltz
prediction fitted with ChimeraX `fitmap`. There is no ChimeraX in Colab, so this cell does both parts here:

0. **Prepare the inputs** — CryoBoltz indexes the aligned model by `label_seq_id`. PDB files don't have it
   (gemmi leaves it unset, and the run dies with a `TypeError`), and a ligand or metal sitting *inside* a
   protein chain shifts every later chain's residue index. This cell rewrites the file as mmCIF with
   `label_seq_id`, and strips waters, hydrogens, altlocs and (optionally) ligands. It also writes a
   protein-only copy of the *sequence* file if that lists ligands or nucleic acids, since guidance's forward
   model cannot handle them.
1. **Unguided prediction** — plain Boltz-1, no map.
2. **Rigid-body fit** — random orientation search from the density centroid, scored by the mean map value at
   CA positions, then Powell refinement. On synthetic tests it recovers a 40 Å / 90° displacement to <0.2 Å
   in a few seconds, but it is a crude scorer: **look at the overlay it prints** before you trust it. Highly
   symmetric maps, or a prediction whose conformation is very different from the map, can trap it in the wrong pose.

There is also a **handedness test**: fit the model into the map and into its z-flip and compare the scores.
A cryoDRGN reconstruction of the wrong hand cannot be fitted by a correct-handed model, and at moderate
resolution the mistake is easy to miss by eye. If the flipped map wins, apply `FLIP_AXIS = 'z'` in Step 5.

Step 0 above is cheap and safe, so it is on by default; the others are opt-in. Skip the cell entirely if
you already have a prepared, aligned mmCIF (the Pma1 demo does).

In [ ]:
#@title Step 6 · Unguided prediction and/or rigid-body fit { display-mode: "form" }
#@markdown Sanitise whatever `CIF_PATH` points at: drop waters/hydrogens/altlocs, optionally drop
#@markdown ligands and metals, and write mmCIF **with `label_seq_id`** — which is how CryoBoltz indexes
#@markdown residues, and which PDB files never carry. Leave this on if Step 4 complained about the model.
PREPARE_ALIGNED_MODEL = True #@param {type:"boolean"}
STRIP_LIGANDS_AND_METALS = True #@param {type:"boolean"}
LABEL_SEQ_FROM = "auto" #@param ["auto", "seqres", "auth"]
#@markdown Also write a protein-only copy of the **sequence** file when it lists ligands, metals or nucleic
#@markdown acids — guidance cannot model those, so a guided run needs them gone from the input too.
DROP_NON_PROTEIN_FROM_SEQUENCE = True #@param {type:"boolean"}
#@markdown ---
MAKE_UNGUIDED_PREDICTION = False #@param {type:"boolean"}
#@markdown Fit the model into the map **and** into its z-flip and compare — a cheap check for a
#@markdown reconstruction of the wrong handedness. Apply the winning flip in Step 5.
HANDEDNESS_TEST = False #@param {type:"boolean"}
RIGID_FIT_TO_MAP = False #@param {type:"boolean"}
FIT_ROTATIONS = 1500 #@param {type:"integer"}
FIT_SEED = 0 #@param {type:"integer"}
SHOW_OVERLAY = True #@param {type:"boolean"}

require("SEQ_PATH", "MAP_PATH", "CACHE_DIR", "INPUT_DIR", "BOLTZ_CMD")
from pathlib import Path

if MAKE_UNGUIDED_PREDICTION:
    out = Path(WORK_DIR) / "unguided"
    stem = Path(SEQ_PATH).stem
    model0 = out / f"boltz_results_{stem}/predictions/{stem}/{stem}_model_0.cif"
    cmd = [*BOLTZ_CMD, "predict", str(SEQ_PATH), "--out_dir", str(out), "--cache", str(CACHE_DIR),
           "--diffusion_samples", "1", "--output_format", "mmcif", "--num_workers", "1", "--override"]
    if USE_MSA_SERVER:
        cmd += ["--use_msa_server"]
    print("■ unguided Boltz prediction (no map) — this is a full Boltz-1 run\n")
    if run_streaming(cmd, cwd=REPO_DIR, log_path=Path(WORK_DIR) / "logs/unguided.log") != 0 or not model0.exists():
        raise RuntimeError("the unguided prediction failed — see the log above")
    CIF_PATH = model0.resolve()
    print(f"\n✔ CIF_PATH = {CIF_PATH}")
    print("  Note: an unguided prediction is in an arbitrary frame — tick RIGID_FIT_TO_MAP to place it.")

if PREPARE_ALIGNED_MODEL and globals().get("CIF_PATH"):
    from pathlib import Path
    src = Path(CIF_PATH)
    raw = raw_structure_report(src)
    needs_work = bool(raw["label_seq_unset"] or raw["het_before_protein"] or raw["gaps"]
                      or src.suffix.lower() in (".pdb", ".ent"))
    if needs_work:
        print("■ preparing the aligned model for CryoBoltz's guidance loader")
        CIF_PATH = prepare_aligned_model(src, Path(INPUT_DIR) / (src.stem + "_prepared.cif"),
                                         strip_ligands=STRIP_LIGANDS_AND_METALS,
                                         label_seq_from=LABEL_SEQ_FROM).resolve()
        print(f"\n✔ CIF_PATH = {CIF_PATH}")
        print("  Re-run Step 4 to confirm CryoBoltz's own loader is happy with it.")
    else:
        print(f"■ {src.name} already carries label_seq_id and has no ligands ahead of protein chains — "
              "nothing to prepare")

if DROP_NON_PROTEIN_FROM_SEQUENCE and globals().get("SEQ_PATH"):
    from pathlib import Path
    others = [(c, t) for c, t, _ in parse_sequence_file(SEQ_PATH) if t != "protein"]
    if others:
        src = Path(SEQ_PATH)
        print(f"■ {src.name} lists non-protein entities: {', '.join(f'{c} ({t})' for c, t in others)}")
        dst = Path(INPUT_DIR) / (src.stem + "_protein_only" + src.suffix)
        kept, dropped = write_protein_only_sequence(src, dst)
        SEQ_PATH = dst.resolve()
        print(f"    kept chains {kept}, dropped {dropped} → {dst.name}")
        print(f"\n✔ SEQ_PATH = {SEQ_PATH}")
        print("  Those entities are simply not predicted now. If you need them modelled, run plain Boltz-1")
        print("  (no --density_map) on the original file and use this guided run for the protein.")
        USE_MSA_SERVER = not has_precomputed_msa(SEQ_PATH)

if HANDEDNESS_TEST:
    require("CIF_PATH", "MAP_PATH", "THRESH")
    from pathlib import Path
    info = MapInfo(MAP_PATH)
    zflip = flip_map(MAP_PATH, Path(INPUT_DIR) / f"{Path(MAP_PATH).stem}_zflip.mrc", axis="z")
    finfo = MapInfo(zflip)
    ca = np.vstack([v["ca"] for v in cif_chains(CIF_PATH).values() if len(v["ca"])])
    print(f"\n■ handedness test on {Path(MAP_PATH).name} ({FIT_ROTATIONS} orientations per hand)")
    print("  A mirror leaves the map's value distribution untouched, so the two fit scores are directly")
    print("  comparable. Fits are from scratch, so this does not depend on your current alignment.")
    hands = {}
    for label, target in (("as given", info), ("z-flipped", finfo)):
        _, _, st = rigid_fit(CIF_PATH, target, thresh=THRESH, n_rotations=FIT_ROTATIONS,
                             seed=FIT_SEED, verbose=False)
        hands[label] = st
    show_table([{"map": label, "best fit score": round(st["score"], 4),
                 "CA in density": f"{100 * st['ca_inside_density']:.1f}%"} for label, st in hands.items()]
               + [{"map": "your current pose", "best fit score": round(float(info.sample(ca).mean()), 4),
                   "CA in density": f"{100 * float((info.sample(ca) > THRESH).mean()):.1f}%"}])
    a, b = hands["as given"]["score"], hands["z-flipped"]["score"]
    if b > 1.05 * a:
        print(f"  → the z-flipped map fits {100 * (b / a - 1):.0f}% better: this reconstruction is probably of the "
              "wrong hand. Set FLIP_AXIS = 'z' in Step 5, re-run it, then come back and RIGID_FIT_TO_MAP.")
    elif a > 1.05 * b:
        print(f"  → the map as given fits {100 * (a / b - 1):.0f}% better — handedness looks right, no flip needed.")
    else:
        print("  → the two hands score within 5% of each other, so this test cannot separate them (expected at "
              "low resolution or for a globular envelope). Judge it on helix chirality in the fitted model, or "
              "run both and compare the guidance loss and map-model agreement.")

if RIGID_FIT_TO_MAP:
    require("CIF_PATH", "THRESH")
    info = MapInfo(MAP_PATH)
    print(f"\n■ rigid-body fitting {Path(CIF_PATH).name} into {Path(MAP_PATH).name} "
          f"({FIT_ROTATIONS} orientations, threshold {THRESH:g})")
    before_inside = float((info.sample(np.vstack([v["ca"] for v in cif_chains(CIF_PATH).values()
                                                  if len(v["ca"])])) > THRESH).mean())
    rot, trans, stats = rigid_fit(CIF_PATH, info, thresh=THRESH, n_rotations=FIT_ROTATIONS, seed=FIT_SEED)
    fitted = Path(INPUT_DIR) / (Path(CIF_PATH).stem + "_fitted.cif")
    transform_structure(CIF_PATH, fitted, rot, trans)
    moved = ca_rmsd(CIF_PATH, fitted)
    CIF_PATH = fitted.resolve()
    print(f"  moved the model by {moved:.1f} Å RMSD")
    print(f"  CA atoms in density: {100 * before_inside:.1f}% → {100 * stats['ca_inside_density']:.1f}%")
    print(f"\n✔ CIF_PATH = {CIF_PATH}")
    if stats["ca_inside_density"] < 0.6:
        print("⚠️  Less than 60% of CA atoms ended up in density. Inspect the overlay; try more rotations, a "
              "different threshold, or fit in ChimeraX/Phenix and upload the result.")

if SHOW_OVERLAY and CIF_PATH:
    print("\n■ overlay — the initial model must sit inside the density before you run CryoBoltz")
    isosurface_figure(MapInfo(MAP_PATH), {"initial model": CIF_PATH}, iso=THRESH,
                      title="Initial structure vs map").show()
elif not (MAKE_UNGUIDED_PREDICTION or RIGID_FIT_TO_MAP or HANDEDNESS_TEST or PREPARE_ALIGNED_MODEL):
    print("Nothing selected — using CIF_PATH as it is:", globals().get("CIF_PATH"))

## Step 7 · Run CryoBoltz

This is the `boltz predict` command from the README, wired to everything the previous steps produced. The
cell prints the exact shell command before running it, so you can copy it to a cluster verbatim.

### The guidance knobs

| Option | Default | What it does / when to change it |
|--------|---------|----------------------------------|
| `--global_steps` | `101 150` | Diffusion steps (1-indexed, of `--sampling_steps`) where the **global** term pulls the structure onto a point-cloud version of the map. Start it later if the prediction is being dragged into noise. |
| `--global_scale` | `0.25 0.05` | Guidance strength at the start / end of the global phase (must be non-increasing). Raise if the model never enters the density; lower if the structure distorts. |
| `--local_steps` | `151 175` | Steps where the **local** term matches a simulated map to your map, voxel by voxel. |
| `--local_scale` | `0.5 0.5` | Strength of the local phase. This is what sharpens side-chain-level agreement. |
| `--cloud_size` | `0.25` | Point-cloud density for the global term. Lower it to cut memory and to stay inside geomloss's fast backend. |
| `--dust` | `5` | Drops connected components smaller than this many voxels — cheap noise removal for the global term. |
| `--thresh` | from Step 4 | Voxels below this are zeroed for the global term. |
| `--res` | from Step 3 | Resolution used by the local term's forward model. |
| `--voxel_batch` | `32768` | Voxels processed at once during local guidance. **Lower this first when you hit CUDA OOM.** |
| `--diffusion_samples` | `1` | Independent conformations per run. Memory and time scale with it; the paper uses 5. |
| `--step_scale` | `1.638` | Sampling temperature — lower gives more diverse samples. |
| `--seed` | `0` | Set for reproducibility; `-1` here leaves it unseeded. |

> The step windows are **absolute step numbers**, so they only make sense relative to `--sampling_steps`.
> If you change `--sampling_steps` and leave the windows at their defaults, this cell rescales them for you
> and says so.

In [ ]:
#@title Step 7 · Run the guided prediction { display-mode: "form" }
DIFFUSION_SAMPLES = 1 #@param {type:"slider", min:1, max:10, step:1}
SAMPLING_STEPS = 200 #@param {type:"integer"}
RECYCLING_STEPS = 3 #@param {type:"integer"}
STEP_SCALE = 1.638 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
#@markdown `SEED = -1` runs unseeded.
MSA = "auto" #@param ["auto", "use MMseqs2 server", "input file already has MSAs"]
#@markdown ### Guidance
GLOBAL_GUIDANCE = True #@param {type:"boolean"}
GLOBAL_STEPS = "101 150" #@param {type:"string"}
GLOBAL_SCALE = "0.25 0.05" #@param {type:"string"}
LOCAL_GUIDANCE = True #@param {type:"boolean"}
LOCAL_STEPS = "151 175" #@param {type:"string"}
LOCAL_SCALE = "0.5 0.5" #@param {type:"string"}
CLOUD_SIZE = 0.25 #@param {type:"number"}
DUST = 5 #@param {type:"integer"}
#@markdown `VOXEL_BATCH = 0` uses the value Step 4 suggested for your GPU.
VOXEL_BATCH = 0 #@param {type:"integer"}
#@markdown ### Output
OUTPUT_FORMAT = "mmcif" #@param ["mmcif", "pdb"]
WRITE_GUIDANCE_LOSS = True #@param {type:"boolean"}
WRITE_TRAJECTORY = False #@param {type:"boolean"}
NUM_WORKERS = 2 #@param {type:"integer"}
KEEP_PREVIOUS_RUNS = True #@param {type:"boolean"}

require("SEQ_PATH", "MAP_PATH", "CIF_PATH", "CACHE_DIR", "RES", "THRESH", "BOLTZ_CMD")
import shlex, shutil, time
from datetime import datetime
from pathlib import Path

def _pair(text, cast, label):
    parts = str(text).replace(",", " ").split()
    if len(parts) != 2:
        raise ValueError(f"{label} must be two numbers, e.g. '101 150' (got {text!r})")
    return [cast(p) for p in parts]

def _rescale(text, label):
    """Rescale a default step window when --sampling_steps is not 200."""
    lo, hi = _pair(text, int, label)
    if SAMPLING_STEPS != 200 and text.strip() in ("101 150", "151 175"):
        f = SAMPLING_STEPS / 200.0
        lo, hi = max(1, round(lo * f)), min(SAMPLING_STEPS, round(hi * f))
        print(f"ℹ️  rescaled {label} to '{lo} {hi}' for --sampling_steps {SAMPLING_STEPS}")
    if hi != -1 and not 1 <= lo <= hi <= SAMPLING_STEPS:
        raise ValueError(f"{label} = '{lo} {hi}' is outside 1..{SAMPLING_STEPS}")
    return lo, hi

if not (GLOBAL_GUIDANCE or LOCAL_GUIDANCE):
    raise ValueError("at least one of the guidance terms must be on (that is a CryoBoltz requirement)")

OUT_DIR = Path(WORK_DIR) / "results"
stem = Path(SEQ_PATH).stem
run_root = OUT_DIR / f"boltz_results_{stem}"
pred_root = run_root / "predictions"
archived = None
if KEEP_PREVIOUS_RUNS and pred_root.exists() and any(pred_root.iterdir()):
    archived = run_root / "previous_runs" / datetime.now().strftime("%Y%m%d_%H%M%S")
    archived.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(pred_root), str(archived))
    print(f"■ archived the previous predictions to {archived}\n")

voxel_batch = VOXEL_BATCH or globals().get("SUGGESTED_VOXEL_BATCH", 32768)
use_server = USE_MSA_SERVER if MSA == "auto" else (MSA == "use MMseqs2 server")

cmd = [*BOLTZ_CMD, "predict", str(SEQ_PATH),
       "--out_dir", str(OUT_DIR),
       "--cache", str(CACHE_DIR),
       "--density_map", str(MAP_PATH),
       "--aligned_model", str(CIF_PATH),
       "--res", str(RES),
       "--thresh", str(THRESH),
       "--dust", str(DUST),
       "--cloud_size", str(CLOUD_SIZE),
       "--voxel_batch", str(voxel_batch),
       "--diffusion_samples", str(DIFFUSION_SAMPLES),
       "--sampling_steps", str(SAMPLING_STEPS),
       "--recycling_steps", str(RECYCLING_STEPS),
       "--step_scale", str(STEP_SCALE),
       "--output_format", OUTPUT_FORMAT,
       "--num_workers", str(NUM_WORKERS),
       "--override"]
if SEED >= 0:
    cmd += ["--seed", str(SEED)]
if use_server:
    cmd += ["--use_msa_server"]
if GLOBAL_GUIDANCE:
    lo, hi = _rescale(GLOBAL_STEPS, "GLOBAL_STEPS")
    a, b = _pair(GLOBAL_SCALE, float, "GLOBAL_SCALE")
    if a < b:
        raise ValueError("GLOBAL_SCALE must be non-increasing, e.g. '0.25 0.05'")
    cmd += ["--global_steps", str(lo), str(hi), "--global_scale", str(a), str(b)]
else:
    cmd += ["--no_global"]
if LOCAL_GUIDANCE:
    lo, hi = _rescale(LOCAL_STEPS, "LOCAL_STEPS")
    a, b = _pair(LOCAL_SCALE, float, "LOCAL_SCALE")
    if a < b:
        raise ValueError("LOCAL_SCALE must be non-increasing, e.g. '0.5 0.5'")
    cmd += ["--local_steps", str(lo), str(hi), "--local_scale", str(a), str(b)]
else:
    cmd += ["--no_local"]
if WRITE_GUIDANCE_LOSS:
    cmd += ["--write_guidance_loss"]
if WRITE_TRAJECTORY:
    cmd += ["--write_traj"]

print("■ command (copy-paste ready):\n")
print("  " + shlex.join(cmd) + "\n")
print(f"■ inputs   : {Path(SEQ_PATH).name} · {Path(MAP_PATH).name} ({MapInfo(MAP_PATH).n_voxels:,} voxels) · "
      f"{Path(CIF_PATH).name}")
print(f"■ outputs  : {run_root}")
print(f"■ live log : {Path(WORK_DIR) / 'logs/cryoboltz_run.log'}")
print("\nWatch for `Started global guidance` / `Started local guidance` in the log — that is guidance kicking in.")
print("Keep this browser tab alive; Colab stops idle sessions.\n")
print("─" * 78)

t0 = time.time()
rc = run_streaming(cmd, cwd=REPO_DIR, log_path=Path(WORK_DIR) / "logs/cryoboltz_run.log")
elapsed = time.time() - t0
print("\n" + "─" * 78)
print(f"exit code {rc} · {elapsed / 60:.1f} min "
      f"({elapsed / max(SAMPLING_STEPS, 1):.2f} s per sampling step, {DIFFUSION_SAMPLES} sample(s))")

PRED_DIR = pred_root / stem
if rc == 0 and PRED_DIR.exists():
    made = sorted(PRED_DIR.glob(f"*_model_*.{'cif' if OUTPUT_FORMAT == 'mmcif' else 'pdb'}"))
    print(f"✔ {len(made)} structure(s) in {PRED_DIR}")
    for p in made:
        print("   ", p.name)
else:
    print("❌ no predictions were written. Common causes:")
    print("   • CUDA out of memory → lower --voxel_batch / --diffusion_samples, crop or bin the map (Step 5),")
    print("     or switch to a bigger GPU;  • 'No predictions to run, exiting.' → an earlier run is still in")
    print("     results/ (this cell passes --override, so check the log);  • MSA server timeout → re-run.")
    print("   The full log is at", Path(WORK_DIR) / "logs/cryoboltz_run.log")
    if archived:
        print(f"   Your previous predictions are safe in {archived}")

### Step 7b · *Batch:* one run per volume (cryoDRGN ensembles)

Fitting an **ensemble** is the point of CryoBoltz, and cryoDRGN hands you one. This cell walks `VOLUME_LIST`
from Step 3b and runs the guided prediction once per volume, reusing everything that does not depend on the
map: the same sequence, the same aligned model, and — by copying the first run's `processed/` directory — the
**same MSA**, so the MMseqs2 server is queried once rather than N times.

Per volume it recomputes the contour threshold (cryoDRGN volumes are not on a common scale) and crops with
your Step 5 settings, then collects a summary table. Budget accordingly: this is *N* full CryoBoltz runs, so
start with two or three volumes to calibrate the wall clock before launching twenty.

Settings come from Step 7 — run that cell first (even if you interrupt the single run) so the guidance
parameters are defined.

In [ ]:
#@title Step 7b · Run over every volume in VOLUME_LIST { display-mode: "form" }
FIRST_N_VOLUMES = 0 #@param {type:"integer"}
CROP_EACH_VOLUME = True #@param {type:"boolean"}
CROP_PAD_VOXELS = 10 #@param {type:"integer"}
REUSE_FIRST_MSA = True #@param {type:"boolean"}
STOP_ON_FAILURE = False #@param {type:"boolean"}

require("VOLUME_LIST", "SEQ_PATH", "CIF_PATH", "CACHE_DIR", "RES", "BOLTZ_CMD", "SAMPLING_STEPS")
import json, shlex, shutil, time
from pathlib import Path

volumes = list(VOLUME_LIST)[: FIRST_N_VOLUMES] if FIRST_N_VOLUMES > 0 else list(VOLUME_LIST)
stem = Path(SEQ_PATH).stem
batch_root = Path(WORK_DIR) / "results_batch"
batch_root.mkdir(parents=True, exist_ok=True)
n_res = sum(len(s) for _, t, s in parse_sequence_file(SEQ_PATH) if t == "protein")

# guidance settings are taken from Step 7's form, so the two cells cannot drift apart
def guidance_flags():
    flags = []
    if GLOBAL_GUIDANCE:
        lo, hi = _rescale(GLOBAL_STEPS, "GLOBAL_STEPS")
        a, b = _pair(GLOBAL_SCALE, float, "GLOBAL_SCALE")
        flags += ["--global_steps", str(lo), str(hi), "--global_scale", str(a), str(b)]
    else:
        flags += ["--no_global"]
    if LOCAL_GUIDANCE:
        lo, hi = _rescale(LOCAL_STEPS, "LOCAL_STEPS")
        a, b = _pair(LOCAL_SCALE, float, "LOCAL_SCALE")
        flags += ["--local_steps", str(lo), str(hi), "--local_scale", str(a), str(b)]
    else:
        flags += ["--no_local"]
    return flags

print(f"■ {len(volumes)} volume(s) × {DIFFUSION_SAMPLES} sample(s), {SAMPLING_STEPS} sampling steps each")
print(f"■ outputs under {batch_root}\n")

summary, first_processed = [], None
for i, vol in enumerate(volumes):
    t0 = time.time()
    tag = Path(vol).stem
    out_dir = batch_root / tag
    print("═" * 78)
    print(f"[{i + 1}/{len(volumes)}] {tag}")
    info = MapInfo(vol)
    thresh = pick_threshold(info, expected_volume(n_res), verbose=False)
    map_for_run = Path(vol)
    if CROP_EACH_VOLUME:
        try:
            map_for_run = crop_map(vol, Path(INPUT_DIR) / f"{tag}_cropped.mrc", REPO_DIR,
                                   thresh=thresh, pad=CROP_PAD_VOXELS, verbose=False)
        except Exception as exc:  # noqa: BLE001
            print(f"  ⚠️  crop failed ({exc}); using the full box")
    cropped_info = MapInfo(map_for_run)
    ca = np.vstack([v["ca"] for v in cif_chains(CIF_PATH).values() if len(v["ca"])])
    inside_before = float((cropped_info.sample(ca) > thresh).mean())
    print(f"  threshold {thresh:g} · {info.n_voxels:,} → {cropped_info.n_voxels:,} voxels · "
          f"initial model in density {100 * inside_before:.1f}%")

    if REUSE_FIRST_MSA and first_processed and first_processed.exists():
        target = out_dir / f"boltz_results_{stem}" / "processed"
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(first_processed, target)
            print("  ✔ reused the MSA/features from the first volume")

    cmd = [*BOLTZ_CMD, "predict", str(SEQ_PATH), "--out_dir", str(out_dir), "--cache", str(CACHE_DIR),
           "--density_map", str(map_for_run), "--aligned_model", str(CIF_PATH),
           "--res", str(RES), "--thresh", str(thresh), "--dust", str(DUST),
           "--cloud_size", str(CLOUD_SIZE), "--voxel_batch", str(voxel_batch),
           "--diffusion_samples", str(DIFFUSION_SAMPLES), "--sampling_steps", str(SAMPLING_STEPS),
           "--recycling_steps", str(RECYCLING_STEPS), "--step_scale", str(STEP_SCALE),
           "--output_format", OUTPUT_FORMAT, "--num_workers", str(NUM_WORKERS), "--override"]
    if SEED >= 0:
        cmd += ["--seed", str(SEED)]
    if use_server and not (REUSE_FIRST_MSA and first_processed):
        cmd += ["--use_msa_server"]
    cmd += guidance_flags()
    if WRITE_GUIDANCE_LOSS:
        cmd += ["--write_guidance_loss"]
    print("  $", shlex.join(cmd))
    rc = run_streaming(cmd, cwd=REPO_DIR, log_path=Path(WORK_DIR) / f"logs/batch_{tag}.log")

    pred_dir = out_dir / f"boltz_results_{stem}" / "predictions" / stem
    row = {"#": i, "volume": tag, "thresh": thresh, "voxels": cropped_info.n_voxels,
           "minutes": round((time.time() - t0) / 60, 1), "exit": rc, "models": 0,
           "confidence": float("nan"), "CA in density": f"{100 * inside_before:.0f}% → n/a"}
    if rc == 0 and pred_dir.exists():
        models = sorted(pred_dir.glob(f"*_model_*.{'cif' if OUTPUT_FORMAT == 'mmcif' else 'pdb'}"))
        row["models"] = len(models)
        if models:
            best = models[0]
            conf = pred_dir / f"confidence_{best.stem}.json"
            if conf.exists():
                row["confidence"] = round(json.loads(conf.read_text()).get("confidence_score", float("nan")), 4)
            fitted = np.vstack([v["ca"] for v in cif_chains(best).values() if len(v["ca"])])
            after = float((cropped_info.sample(fitted) > thresh).mean())
            row["CA in density"] = f"{100 * inside_before:.0f}% → {100 * after:.0f}%"
        if first_processed is None:
            candidate = out_dir / f"boltz_results_{stem}" / "processed"
            if candidate.exists():
                first_processed = candidate
    else:
        print(f"  ❌ exit {rc}; log: {Path(WORK_DIR) / f'logs/batch_{tag}.log'}")
        if STOP_ON_FAILURE:
            summary.append(row)
            break
    summary.append(row)
    print(f"  done in {row['minutes']} min · CA in density {row['CA in density']}")

print("\n" + "═" * 78)
print("■ batch summary")
show_table(summary)
BATCH_ROOT = batch_root
ok = [r for r in summary if r["exit"] == 0 and r["models"]]
print(f"\n{len(ok)}/{len(summary)} volume(s) produced structures. Set PRED_DIR to one of the directories under")
print(f"{batch_root}/<volume>/boltz_results_{stem}/predictions/{stem} to inspect it in Steps 8 and 9, e.g.:")
if ok:
    print(f"  PRED_DIR = Path('{batch_root}/{ok[0]['volume']}/boltz_results_{stem}/predictions/{stem}')")
print("\nThe fitted conformations across volumes are the ensemble — compare them to each other, not just to")
print("their own map (`ca_rmsd(model_a, model_b)` is available).")

## Step 8 · Confidence scores and guidance diagnostics

Predictions are written as `<name>_model_<rank>.cif`, ranked 0 = most confident. Alongside each one:
`confidence_*.json` (summary scores), `plddt_*.npz` (per-residue pLDDT), and — with `--write_guidance_loss`
— a single `guidance_loss.npz` holding the loss curves per diffusion step and sample.

Confidence scores come from Boltz-1 and say nothing about **map agreement**, so use them as a tie-breaker
only. Judge the fit visually (Step 9) and, for anything quantitative, with a real validation package
([Phenix](https://phenix-online.org/documentation/reference/validation_cryo_em.html), MolProbity, `Q`-scores).

In [ ]:
#@title Step 8 · Scores + guidance loss curves { display-mode: "form" }
require("PRED_DIR")
import json
from pathlib import Path

import numpy as np

PRED_DIR = Path(PRED_DIR)
rows = []
for struct in sorted(list(PRED_DIR.glob("*_model_*.cif")) + list(PRED_DIR.glob("*_model_*.pdb"))):
    rank = int(struct.stem.split("_model_")[-1])
    row = {"rank": rank, "file": struct.name}
    conf = PRED_DIR / f"confidence_{struct.stem}.json"
    if conf.exists():
        d = json.loads(conf.read_text())
        row.update({"confidence": round(d.get("confidence_score", float("nan")), 4),
                    "complex_plddt": round(d.get("complex_plddt", float("nan")), 4),
                    "ptm": round(d.get("ptm", float("nan")), 4),
                    "iptm": round(d.get("iptm", float("nan")), 4)})
    plddt = PRED_DIR / f"plddt_{struct.stem}.npz"
    if plddt.exists():
        arr = np.load(plddt)["plddt"]
        row["mean_plddt"] = round(float(arr.mean()), 4)
        row["low_plddt_res"] = int((arr < 0.5).sum())
    rows.append(row)

if not rows:
    raise RuntimeError(f"no *_model_*.cif/.pdb files in {PRED_DIR} — did Step 7 finish successfully? "
                       "Earlier runs may have been archived to ../previous_runs/")
print(f"■ {len(rows)} prediction(s) in {PRED_DIR}\n")
show_table(sorted(rows, key=lambda r: r["rank"]))
print("\n  rank 0 = highest Boltz confidence. `low_plddt_res` counts residues with pLDDT < 0.5.")

loss_file = PRED_DIR / "guidance_loss.npz"
if loss_file.exists():
    import matplotlib.pyplot as plt
    losses = np.load(loss_file)
    keys = [k for k in losses.files]
    fig, axes = plt.subplots(1, len(keys), figsize=(5.5 * len(keys), 3.4), squeeze=False)
    for ax, key in zip(axes[0], keys):
        curves = np.atleast_2d(losses[key])          # (sample, step), -1 where the term was inactive
        for i, curve in enumerate(curves):
            steps = np.arange(1, curve.shape[0] + 1)
            active = curve >= 0
            ax.plot(steps[active], curve[active], lw=1.6, label=f"sample {i}")
        ax.set(xlabel="diffusion step", ylabel=f"{key} guidance loss", title=f"{key} guidance")
        if curves.shape[0] > 1:
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print("A loss that falls and then flattens is guidance doing its job. A flat or rising curve means the")
    print("term never engaged: check the step windows, the scale, and the threshold.")
else:
    print("\n(no guidance_loss.npz — tick WRITE_GUIDANCE_LOSS in Step 7 to get the curves)")

# per-residue pLDDT of the best model
best = sorted(rows, key=lambda r: r["rank"])[0] if rows else None
if best:
    plddt = PRED_DIR / f"plddt_{Path(best['file']).stem}.npz"
    if plddt.exists():
        import matplotlib.pyplot as plt
        arr = np.load(plddt)["plddt"]
        fig, ax = plt.subplots(figsize=(9, 2.6))
        ax.plot(np.arange(1, len(arr) + 1), arr, lw=1.2, color="#2e86ab")
        ax.axhline(0.5, color="#e4572e", ls="--", lw=1)
        ax.set(xlabel="residue", ylabel="pLDDT", ylim=(0, 1), title=f"per-residue pLDDT — {best['file']}")
        plt.tight_layout()
        plt.show()

## Step 9 · Look at the structures

Two views, because they answer different questions:

1. **Cartoon viewer** (py3Dmol) — is the *fold* sensible? Colouring by pLDDT (stored in the B-factor column
   of the output) shows which regions the model is unsure about.
2. **Model-in-map overlay** (Plotly) — does it *fit the density*? Guided predictions are written back in the
   map's coordinate frame, so nothing needs to be superposed. (Unguided predictions are **not**, which is why
   they need Step 6.)

In [ ]:
#@title Step 9a · Cartoon viewer (colour by pLDDT / chain / model) { display-mode: "form" }
MODELS_TO_SHOW = "0" #@param ["0", "0,1", "all"] {allow-input: true}
COLOUR_BY = "pLDDT" #@param ["pLDDT", "chain", "model"]
SHOW_INITIAL_MODEL = False #@param {type:"boolean"}

require("PRED_DIR")
from pathlib import Path

import py3Dmol

structs = sorted(list(Path(PRED_DIR).glob("*_model_*.cif")) + list(Path(PRED_DIR).glob("*_model_*.pdb")),
                 key=lambda p: int(p.stem.split("_model_")[-1]))
if not structs:
    raise RuntimeError(f"no predictions in {PRED_DIR} — run Step 7 first")
if MODELS_TO_SHOW.strip().lower() == "all":
    chosen = structs
else:
    want = {int(x) for x in MODELS_TO_SHOW.replace(" ", "").split(",") if x != ""}
    chosen = [p for p in structs if int(p.stem.split("_model_")[-1]) in want] or structs[:1]

view = py3Dmol.view(width=900, height=620)
PALETTE = ["#e4572e", "#17bebb", "#ffc914", "#76b041", "#a06cd5", "#2e86ab"]
for i, path in enumerate(chosen):
    view.addModel(Path(path).read_text(), "cif" if path.suffix == ".cif" else "pdb")
    if COLOUR_BY == "pLDDT":
        # the writer stores pLDDT × 100 in the B-factor column
        style = {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}}
    elif COLOUR_BY == "chain":
        style = {"cartoon": {"colorscheme": "chain"}}
    else:
        style = {"cartoon": {"color": PALETTE[i % len(PALETTE)]}}
    view.setStyle({"model": i}, style)
INITIAL = globals().get("CIF_PATH")
if SHOW_INITIAL_MODEL and INITIAL:
    view.addModel(Path(INITIAL).read_text(), "cif")
    view.setStyle({"model": len(chosen)}, {"cartoon": {"color": "#9aa5b1", "opacity": 0.6}})
view.zoomTo()
print("showing:", ", ".join(p.name for p in chosen)
      + (f"  (+ initial model {Path(INITIAL).name} in grey)" if SHOW_INITIAL_MODEL and INITIAL else ""))
if COLOUR_BY == "pLDDT":
    print("colour: red = pLDDT 50 or below → blue = 90 or above")
view.show()

In [ ]:
#@title Step 9b · Model-in-map overlay { display-mode: "form" }
OVERLAY_MODELS = "0" #@param ["0", "0,1,2", "all"] {allow-input: true}
#@markdown `ISO_LEVEL = 0` contours the map at `THRESH`.
ISO_LEVEL = 0 #@param {type:"number"}
SURFACE_OPACITY = 0.22 #@param {type:"slider", min:0.05, max:0.8, step:0.01}
#@markdown Grid points per axis for the isosurface. Higher = smoother surface, heavier notebook.
MAX_GRID = 48 #@param {type:"slider", min:24, max:96, step:8}
SHOW_INITIAL_MODEL = True #@param {type:"boolean"}

require("PRED_DIR", "MAP_PATH", "THRESH")
from pathlib import Path

structs = sorted(list(Path(PRED_DIR).glob("*_model_*.cif")) + list(Path(PRED_DIR).glob("*_model_*.pdb")),
                 key=lambda p: int(p.stem.split("_model_")[-1]))
if not structs:
    raise RuntimeError(f"no predictions in {PRED_DIR} — run Step 7 first")
if OVERLAY_MODELS.strip().lower() == "all":
    chosen = structs
else:
    want = {int(x) for x in OVERLAY_MODELS.replace(" ", "").split(",") if x != ""}
    chosen = [p for p in structs if int(p.stem.split("_model_")[-1]) in want] or structs[:1]

models = {f"model {int(p.stem.split('_model_')[-1])}": p for p in chosen}
INITIAL = globals().get("CIF_PATH")
if SHOW_INITIAL_MODEL and INITIAL:
    models["initial"] = INITIAL

info = MapInfo(MAP_PATH)
iso = ISO_LEVEL or THRESH
print(f"map {Path(MAP_PATH).name} contoured at {iso:g}; CA traces for {', '.join(models)}")
for label, path in models.items():
    ca = np.vstack([v["ca"] for v in cif_chains(path).values() if len(v["ca"])])
    print(f"  {label:12s} {100 * float((info.sample(ca) > iso).mean()):5.1f}% of CA atoms inside the contour")
isosurface_figure(info, models, iso=iso, max_grid=MAX_GRID, opacity=SURFACE_OPACITY,
                  title=f"{Path(MAP_PATH).name} @ {iso:g}").show()

## Step 10 · Save your results

Colab throws the VM away when the session ends. Download the results or copy them to Drive.

In [ ]:
#@title Step 10 · Zip and download / copy to Drive { display-mode: "form" }
DOWNLOAD_ZIP = True #@param {type:"boolean"}
COPY_TO_DRIVE = False #@param {type:"boolean"}
DRIVE_SUBDIR = "cryoboltz_results" #@param {type:"string"}

require("PRED_DIR", "WORK_DIR", "SEQ_PATH", "MAP_PATH")
import shutil
from datetime import datetime
from pathlib import Path

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
bundle = Path(WORK_DIR) / f"cryoboltz_{stamp}"
(bundle / "predictions").mkdir(parents=True, exist_ok=True)
shutil.copytree(PRED_DIR, bundle / "predictions", dirs_exist_ok=True)
for src, name in ((SEQ_PATH, None), (globals().get("CIF_PATH"), "initial_model.cif"), (MAP_PATH, None)):
    if src and Path(src).exists():
        shutil.copy(src, bundle / (name or Path(src).name))
log = Path(WORK_DIR) / "logs/cryoboltz_run.log"
if log.exists():
    shutil.copy(log, bundle / "cryoboltz_run.log")

archive = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
print(f"■ {archive}  ({human(Path(archive).stat().st_size)})")
print("   contents: predictions/ (structures, confidence, pLDDT, guidance loss), the inputs, and the run log")

if COPY_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(archive, dest)
        print(f"✔ copied to {dest / Path(archive).name}")
    except Exception as exc:  # noqa: BLE001
        print(f"⚠️  Drive copy failed: {exc}")

if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:  # noqa: BLE001
        print(f"⚠️  automatic download failed ({exc}); grab it from the Files pane on the left")

---

## Troubleshooting

| Symptom | What to do |
|---------|-----------|
| `torch.OutOfMemoryError` / `CUDA out of memory` | In order: lower `--voxel_batch` (Step 7), crop **and** bin the map (Step 5), `--diffusion_samples 1`, shorten the local-guidance window, then use a bigger GPU. Guidance runs with gradients enabled in float32, so it needs noticeably more memory than plain Boltz-1. |
| `No predictions to run, exiting.` | A prediction for this input already exists in `results/`. Step 7 passes `--override` and archives previous runs, so re-run that cell rather than the CLI by hand. |
| `module compiled against NumPy 2.x cannot be run in NumPy 1.x` | The install pins `numpy==1.26.3`. `Runtime ▸ Restart session`, then re-run Steps 1, 1b and onwards — nothing is re-downloaded. |
| `TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'` when guidance starts | Your aligned model is a **PDB file** (or any file without `label_seq_id`) — CryoBoltz indexes the model by `label_seq_id`. Turn on `PREPARE_ALIGNED_MODEL` in Step 6, or convert with `gemmi convert --to=mmcif model.pdb model.cif`. |
| `IndexError: list index out of range` when guidance starts | A ligand, metal or other non-amino-acid residue sits inside a protein chain of the aligned model. Chain offsets are computed from the *total* residue count of the previous chain, so those residues shift every later chain. Strip them (Step 6, `STRIP_LIGANDS_AND_METALS`). |
| `WARNING: <RES> in structure does not match <RES> in sequence` | Your aligned model and sequence file disagree. Check chain **order** and **labels**, and that residue numbering (`label_seq_id`) starts at 1 with no shift. Step 4 flags this before you burn GPU time. |
| Prediction ends up nowhere near the density | The aligned model was not actually aligned, or the map origin is wrong (`nstart` vs `origin` — see Step 4/5). Fit it with Step 6 and check the overlay. |
| `KeyError` on an element during guidance | The guidance forward model only knows H, C, N, O, S. Nucleic acids, metals and most ligands cannot be guided — predict them if you like, but guide against protein density. |
| PyKeOps tries to compile CUDA kernels, or fails to | geomloss switches to its PyKeOps backend once `atoms × cloud points` exceeds 5000². Bin the map (Step 5) or lower `--cloud_size` to stay on the fast tensorized path. |
| cryoDRGN volume claims **1.0 Å/px** | `cryodrgn eval_vol --Apix` defaults to 1.0, and `analyze` falls back to it when the CTF pixel size is ambiguous. Set the true value in Step 3b (`original_Apix × original_box / cryodrgn_box`); Step 3b also cross-checks it against your model's radius of gyration. |
| A model fits no orientation of a cryoDRGN volume | The handedness may be flipped. Run the **handedness test** in Step 6, then set `FLIP_AXIS = "z"` in Step 5 — that is bit-for-bit what `cryodrgn eval_vol --flip` does (`vol.flip([0])` on a `[z,y,x]` array). Re-fit afterwards: a flip moves the density to the other side of the box, so an existing alignment no longer applies. |
| Dozens of cryoDRGN volumes to fit | Step 3b collects them and Step 7b loops, copying the first run's `processed/` so the MSA server is hit once. Start with two or three volumes to calibrate the wall clock. |
| MSA server errors / timeouts | `api.colabfold.com` is rate-limited and shared. Wait and re-run, or precompute the MSA and point to the `.a3m` from your FASTA/YAML (`>A\|protein\|msa.a3m`). |
| Local guidance is glacial | Its cost is (voxels × atoms) per guided step. Cropping tightly is the single biggest win; binning is the second. |
| Session disconnected mid-run | Keep the tab open and active. Park the weight cache and results on Drive (Steps 2 and 10) so a restart is cheap. Colab Pro gives longer sessions and background execution. |
| The fit looks plausible but the conformation looks wrong | Try several `--diffusion_samples`, a lower `--step_scale` for more diversity, and judge candidates by map–model agreement rather than confidence score. |

### A few practical notes

* **Guided outputs live in the map's frame.** CryoBoltz shifts samples back by the aligned model's centre at
  the end of sampling, so predictions overlay the map directly. Unguided predictions do not.
* **Confidence ≠ fit.** `confidence_score` is Boltz-1's own estimate. Rank candidates by map–model agreement
  (real-space CC, Q-scores, Phenix validation) when it matters.
* **Guidance is protein-only.** The forward model uses H/C/N/O/S scattering amplitudes and per-residue atom
  templates; ligand atoms are typed as `UNK` (a 5-atom template), so a system containing ligands or metals
  cannot be guided. Predict those with plain Boltz-1, or guide the protein-only system — Step 6 writes a
  protein-only copy of the sequence file, and strips ligands and metals from the aligned model, by default.
* **Reproducibility.** Pass `--seed`; Step 7 does by default.
* **An already-aligned model stays aligned.** Step 6's preparation only deletes residues and rewrites the
  file — no atom moves — so an alignment you made in ChimeraX against a cryoDRGN volume survives it. Leave
  `RIGID_FIT_TO_MAP` off in that case.

---

## Citation

If you use CryoBoltz, cite both CryoBoltz and Boltz-1:

```bibtex
@article{raghu2025cryoboltz,
  title={Multiscale Guidance of Protein Structure Prediction with Heterogeneous Cryo-EM Data},
  author={Raghu, Rishwanth and Levy, Axel and Wetzstein, Gordon and Zhong, Ellen D.},
  journal={Advances in neural information processing systems},
  year={2025},
}

@article{wohlwend2024boltz1,
  author = {Wohlwend, Jeremy and Corso, Gabriele and Passaro, Saro and Reveiz, Mateo and Leidal, Ken and
            Swiderski, Wojtek and Portnoi, Tally and Chinn, Itamar and Silterra, Jacob and Jaakkola, Tommi and
            Barzilay, Regina},
  title = {Boltz-1: Democratizing Biomolecular Interaction Modeling},
  year = {2024},
  doi = {10.1101/2024.11.19.624167},
  journal = {bioRxiv}
}
```

Questions or bugs in CryoBoltz itself: [open a GitHub issue](https://github.com/ml-struct-bio/cryoboltz/issues).